# SONORA Stage 5 — Cloud GPU Feasibility Benchmark: CLAPSep vs. AudioSep for Ukulele Extraction

> **Experimental Scope & Scientific Honesty Notice**
> - This notebook is an **offline cloud feasibility benchmark** for SONORA Stage 5 (Option B).
> - Neither **CLAPSep** nor **AudioSep** is assumed to work effectively for ukulele-only extraction until actual processed audio outputs are measured and evaluated by human listening.
> - **No perfect separation is claimed.** Both models operate internally at **32,000 Hz mono** (hard 16 kHz bandwidth cap, collapsing stereo to mono) and can introduce audible artifacts.
> - **Real vs. Synthetic separation:** Real-world recordings (Section 4) have **no clean ground truth** and must be evaluated via listening + diagnostic telemetry. Synthetic mixtures (Section 5A) and optional real-speech + synthetic-ukulele mixtures (Section 5B) retain separate target and interference signals so objective metrics can be computed accurately.
> - **Missing Stage 4 reference:** Because the earlier local SONORA temp session expired, **no Stage 4 DSP output is fabricated or substituted**.

---

## Security & Runtime Isolation Requirements (Mandatory)

1. **Fresh Isolated Runtime Only:** Run this notebook ONLY in a fresh, disposable Google Colab runtime with **no mounted sensitive Google Drive files, secrets, or personal credentials**.
2. **No Global Pickle Override:** This notebook does **NOT** set `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1`. Instead, it attempts `torch.load(..., weights_only=True)` first. Where legacy LAION-CLAP (`.pt`) or PyTorch Lightning (`.ckpt`) checkpoints require unrestricted pickle deserialization, it uses a tightly scoped context manager (`scoped_trusted_torch_load`) restricted strictly to exact pre-verified checkpoint file paths (verified by pinned revision, exact byte size, and expected SHA-256), and immediately restores default `torch.load` in `finally:`. User-uploaded audio files are **never** passed to `torch.load`.
3. **Residual Security Risk:** Unrestricted pickle deserialization of `.pt`/`.ckpt` files can execute arbitrary Python code if an upstream artifact was compromised prior to the pinned revision below. Because upstream authors did not publish out-of-band cryptographic signatures (PGP/Sigstore), the SHA-256 checks below verify transport/revision integrity against Hugging Face LFS OIDs — **not** independent cryptographic authentication.

---

## Pinned Immutable Revisions, Sizes, Hashes & Authentication Status

| Model | Pinned Git Commit | Pinned HF Space Revision | Artifact File | Exact Bytes | Expected SHA-256 (HF LFS OID) | Trust / Authentication Status | License |
| :--- | :--- | :--- | :--- | ---: | :--- | :--- | :--- |
| **CLAPSep** | `Aisaka0v0/CLAPSep` @ `bcc768e2064a9bf5a21c0ed0fa23e99fd0cc9aa4` | `AisakaMikoto/CLAPSep` @ `e3c73653fa9aadcddbcc3b5e967182a573589e81` | `model/best_model.ckpt` | `177,818,986` | `6fcc8dbcd7174af86266cf16b4105eced0802352762f81dbfefdce29af3dba04` | **NOT cryptographically authenticated** (same-host HF LFS OID only; no independent hash published in GitHub README) | MIT |
| **CLAPSep (CLAP)** | *(same)* | *(same)* | `model/music_audioset_epoch_15_esc_90.14.pt` | `2,352,471,003` | `fae3e9c087f2909c28a09dc31c8dfcdacbc42ba44c70e972b58c1bd1caf6dedd` | **Cross-matched to upstream `lukewys/laion_clap` (`b3708341...`) on HF; NOT independently signed** | CC0-1.0 |
| **AudioSep** | `Audio-AGI/AudioSep` @ `944583f18b84589dc965de3ad77525c945334252` | `Audio-AGI/AudioSep` @ `0807a713283035d8b0d0628f1b8e3d6580459f7b` | `checkpoint/audiosep_base_4M_steps.ckpt` | `1,264,844,076` | `f8cda01bfd0ebd141eef45d41db7a3ada23a56568465840d3cff04b8010ce82c` | **NOT cryptographically authenticated** (same-host HF LFS OID only; no independent hash published in GitHub README) | MIT |
| **AudioSep (CLAP)** | *(same)* | *(same)* | `checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt` | `2,352,471,003` | `51c68f12f9d7ea25fdaaccf741ec7f81e93ee594455410f3bca4f47f88d8e006` | **Cross-matched to upstream `lukewys/laion_clap` (`b3708341...`) on HF; NOT independently signed** | CC0-1.0 |

### Architectural Comparison Warning (Do Not Assume Feature Parity)
- **CLAPSep** natively supports **positive and negative text queries** (`text_pos`, `text_neg`) via concatenated 1024-dim LAION-CLAP embeddings (`HTSAT-base` LoRA + `HTSAT_Decoder`, magnitude-only mask with mixture phase, 32 kHz mono, 10 s chunks).
- **AudioSep** supports **positive text queries only** (512-dim CLAP embedding conditioning `ResUNet30`, 32 kHz mono). Negative prompts are **NOT** natively supported in AudioSep and are reported as `N/A (unsupported)`.

## Section 1 — Hardware Check (GPU, VRAM, System RAM, Disk & Fail-Closed CPU Policy)

Checks available hardware **before** installing dependencies or downloading checkpoints, and captures the **pre-install PyTorch / CUDA baseline** so Section 2 can abort if `pip` damages the CUDA PyTorch build.
- **Recommended:** NVIDIA CUDA GPU with **≥ 8 GB VRAM** (e.g., Colab T4 15 GB) and **≥ 12 GB system RAM**.
- **Fail-Closed CPU Fallback Policy:** Free Colab GPUs are not guaranteed. CPU fallback is disabled by default (`ALLOW_CPU_FALLBACK = False`). Even when opted in, it **fails closed** if available system RAM cannot be measured OR is `< 10.0 GB`.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

# --- User Policy Toggles ---
ALLOW_CPU_FALLBACK = False  # Set True ONLY if no GPU is allocated AND measured free RAM >= 10 GB
MIN_CPU_FALLBACK_FREE_RAM_GB = 10.0
MIN_GPU_VRAM_GB = 7.5
WORK_ROOT = Path("/content/sonora_stage5_benchmark") if Path("/content").exists() else Path("./sonora_stage5_benchmark").resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)


def measure_system_ram_gb() -> tuple[float | None, float | None, str]:
    """Measure total and available system RAM via psutil or Linux /proc/meminfo. Fails closed if unavailable."""
    try:
        import psutil
        vm = psutil.virtual_memory()
        return vm.total / (1024 ** 3), vm.available / (1024 ** 3), "psutil"
    except ImportError:
        pass

    meminfo = Path("/proc/meminfo")
    if meminfo.is_file():
        vals = {}
        for line in meminfo.read_text(encoding="utf-8", errors="ignore").splitlines():
            parts = line.split()
            if len(parts) >= 2 and parts[0].endswith(":"):
                try:
                    vals[parts[0][:-1]] = int(parts[1])  # kB
                except ValueError:
                    pass
        if "MemTotal" in vals and "MemAvailable" in vals:
            return vals["MemTotal"] / (1024 ** 2), vals["MemAvailable"] / (1024 ** 2), "/proc/meminfo"

    return None, None, "unavailable"


total_ram_gb, avail_ram_gb, ram_source = measure_system_ram_gb()
disk = shutil.disk_usage(WORK_ROOT)
free_disk_gb = disk.free / (1024 ** 3)

try:
    import torchaudio
    pre_torchaudio_ver = torchaudio.__version__
except ImportError:
    pre_torchaudio_ver = None

cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else "None"
gpu_vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)) if cuda_available else 0.0

PRE_INSTALL_TORCH_SNAPSHOT = {
    "torch_version": torch.__version__,
    "torchaudio_version": pre_torchaudio_ver,
    "torch_cuda_version": torch.version.cuda,
    "cuda_available": cuda_available,
}

print("=== RUNTIME ENVIRONMENT REPORT (PRE-INSTALL) ===")
print(f"Python version      : {platform.python_version()} ({sys.executable})")
print(f"OS / Platform       : {platform.platform()}")
print(f"PyTorch (pre)       : {PRE_INSTALL_TORCH_SNAPSHOT['torch_version']} (CUDA build: {PRE_INSTALL_TORCH_SNAPSHOT['torch_cuda_version']})")
print(f"torchaudio (pre)    : {PRE_INSTALL_TORCH_SNAPSHOT['torchaudio_version']}")
print(f"CPU logical cores   : {os.cpu_count()}")
if total_ram_gb is not None and avail_ram_gb is not None:
    print(f"System RAM ({ram_source}): {total_ram_gb:.2f} GB total | {avail_ram_gb:.2f} GB available")
else:
    print("System RAM          : UNMEASURABLE (psutil and /proc/meminfo both unavailable)")
print(f"Free disk space     : {free_disk_gb:.2f} GB at {WORK_ROOT}")
print(f"CUDA available      : {cuda_available}")
print(f"GPU model           : {gpu_name}")
print(f"GPU total VRAM      : {gpu_vram_gb:.2f} GB")

if cuda_available:
    try:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, check=False, timeout=5,
        )
        if smi.returncode == 0 and smi.stdout.strip():
            print(f"nvidia-smi          : {smi.stdout.strip()}")
    except OSError:
        pass

if free_disk_gb < 12.0:
    raise RuntimeError(
        f"Insufficient free disk space ({free_disk_gb:.2f} GB). Need at least 12 GB for repos + ~5.86 GB checkpoints + outputs."
    )

if cuda_available and gpu_vram_gb >= MIN_GPU_VRAM_GB:
    DEVICE = "cuda"
    print(f"\n[OK] Selected execution device: CUDA ({gpu_name}, {gpu_vram_gb:.2f} GB VRAM).")
elif cuda_available:
    DEVICE = "cuda"
    print(
        f"\n[WARNING] GPU VRAM ({gpu_vram_gb:.2f} GB) is below recommended {MIN_GPU_VRAM_GB:.1f} GB. "
        "Chunked sequential execution will be enforced, but OOM risk exists."
    )
else:
    print("\n[WARNING] No NVIDIA CUDA GPU detected.")
    if not ALLOW_CPU_FALLBACK:
        raise RuntimeError(
            "No CUDA GPU detected and ALLOW_CPU_FALLBACK is False. "
            "Switch Colab to Runtime -> Change runtime type -> T4 GPU, OR set ALLOW_CPU_FALLBACK = True "
            "only if measured available RAM >= 10.0 GB."
        )
    # Fail closed if RAM cannot be measured!
    if avail_ram_gb is None:
        raise RuntimeError(
            "CPU fallback FAILED CLOSED: available system RAM could not be measured. "
            "Refusing to load multi-GB checkpoints without verified RAM headroom."
        )
    if avail_ram_gb < MIN_CPU_FALLBACK_FREE_RAM_GB:
        raise RuntimeError(
            f"CPU fallback FAILED CLOSED: only {avail_ram_gb:.2f} GB RAM available "
            f"(minimum {MIN_CPU_FALLBACK_FREE_RAM_GB:.1f} GB required to avoid OOM/swap thrashing)."
        )
    DEVICE = "cpu"
    print(
        f"[CPU FALLBACK ENABLED] Verified {avail_ram_gb:.2f} GB RAM available. "
        "Expect slow offline CPU inference (~1-4x real-time per query)."
    )

HARDWARE_INFO = {
    "python_version": platform.python_version(),
    "pre_install_torch": PRE_INSTALL_TORCH_SNAPSHOT,
    "device": DEVICE,
    "gpu_name": gpu_name,
    "gpu_vram_gb": round(gpu_vram_gb, 2),
    "cpu_cores": os.cpu_count(),
    "total_ram_gb": round(total_ram_gb, 2) if total_ram_gb is not None else None,
    "avail_ram_gb": round(avail_ram_gb, 2) if avail_ram_gb is not None else None,
    "ram_measurement_source": ram_source,
    "free_disk_gb": round(free_disk_gb, 2),
}

## Section 2 — Pinned Repo Checkouts, Dependency Installation & CUDA Integrity Guard

**Nothing is cloned or installed until you explicitly execute this cell.**
- Checks out **exact pinned Git commit hashes** for both repositories and asserts `git rev-parse HEAD` matches:
  - `Aisaka0v0/CLAPSep` @ `bcc768e2064a9bf5a21c0ed0fa23e99fd0cc9aa4`
  - `Audio-AGI/AudioSep` @ `944583f18b84589dc965de3ad77525c945334252`
- Installs compatible dependencies while **explicitly verifying before and after installation** that `torch`, `torchaudio`, and CUDA availability remain intact — **aborting immediately** if `pip` replaces or breaks the CUDA PyTorch build.
- Captures exact Python version, exact `pip freeze` lock snapshot, and pinned Git commits into `environment_manifest.json`.
- **Reproducibility Caveat:** Specifying compatible version ranges in `pip install` does **not** guarantee bit-identical reproducibility across future Google Colab base images; true replay requires the captured `environment_manifest.json` snapshot generated by your run.

In [ ]:
import importlib
import json
import platform
import subprocess
import sys
from pathlib import Path

RUN_INSTALL = True  # Explicit execution guard

PINNED_REPOS = {
    "CLAPSep": {
        "url": "https://github.com/Aisaka0v0/CLAPSep.git",
        "commit": "bcc768e2064a9bf5a21c0ed0fa23e99fd0cc9aa4",
    },
    "AudioSep": {
        "url": "https://github.com/Audio-AGI/AudioSep.git",
        "commit": "944583f18b84589dc965de3ad77525c945334252",
    },
}

REPOS_DIR = WORK_ROOT / "repos"
CLAPSEP_REPO = REPOS_DIR / "CLAPSep"
AUDIOSEP_REPO = REPOS_DIR / "AudioSep"
REPOS_DIR.mkdir(parents=True, exist_ok=True)


def clone_and_pin_repo(name: str, url: str, expected_commit: str, dest: Path) -> str:
    if not dest.exists():
        subprocess.run(["git", "clone", url, str(dest)], check=True)
    subprocess.run(["git", "-C", str(dest), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(dest), "checkout", "--detach", expected_commit], check=True)
    head = subprocess.run(
        ["git", "-C", str(dest), "rev-parse", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    if head != expected_commit:
        raise RuntimeError(f"Git pin mismatch for {name}: got {head}, expected {expected_commit}")
    print(f"[OK] {name:10s} pinned at exact commit {head}")
    return head


if not RUN_INSTALL:
    print("RUN_INSTALL is False — skipping repo cloning and dependency installation.")
else:
    pinned_heads = {}
    pinned_heads["CLAPSep"] = clone_and_pin_repo(
        "CLAPSep", PINNED_REPOS["CLAPSep"]["url"], PINNED_REPOS["CLAPSep"]["commit"], CLAPSEP_REPO
    )
    pinned_heads["AudioSep"] = clone_and_pin_repo(
        "AudioSep", PINNED_REPOS["AudioSep"]["url"], PINNED_REPOS["AudioSep"]["commit"], AUDIOSEP_REPO
    )

    # Pin current torch & torchaudio versions as pip constraints so pip CANNOT replace Colab's CUDA build
    constraints_file = WORK_ROOT / "torch_pin_constraints.txt"
    constraint_lines = [f"torch=={PRE_INSTALL_TORCH_SNAPSHOT['torch_version']}"]
    if PRE_INSTALL_TORCH_SNAPSHOT["torchaudio_version"]:
        constraint_lines.append(f"torchaudio=={PRE_INSTALL_TORCH_SNAPSHOT['torchaudio_version']}")
    constraints_file.write_text("\n".join(constraint_lines) + "\n", encoding="utf-8")

    packages = [
        "numpy>=1.24.0,<2.0.0",
        "scipy>=1.10.0,<1.15.0",
        "soundfile>=0.12.1",
        "librosa>=0.10.1,<0.11.0",
        "torchlibrosa==0.1.0",
        "einops>=0.7.0",
        "loralib==0.1.2",
        "transformers>=4.36.0,<4.46.0",
        "huggingface_hub>=0.20.0",
        "laion-clap==1.1.7",
        "lightning>=2.0.0,<2.5.0",
        "ftfy",
        "braceexpand",
        "webdataset",
        "wget",
        "h5py",
        "psutil",
        "pandas",
        "tabulate",
        "matplotlib",
    ]
    print("Installing dependencies with torch/torchaudio constraints to protect CUDA build...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints_file), *packages],
        check=True,
    )

    # Verify in a FRESH subprocess that torch/torchaudio/CUDA were NOT replaced or broken
    probe_code = (
        "import json, torch; "
        "import torchaudio; "
        "print(json.dumps({"
        "'torch_version': torch.__version__, "
        "'torchaudio_version': torchaudio.__version__, "
        "'torch_cuda_version': torch.version.cuda, "
        "'cuda_available': torch.cuda.is_available()"
        "}))"
    )
    post_probe = subprocess.run(
        [sys.executable, "-c", probe_code],
        capture_output=True, text=True, check=True,
    )
    POST_INSTALL_TORCH_SNAPSHOT = json.loads(post_probe.stdout.strip())
    print("Pre-install PyTorch :", PRE_INSTALL_TORCH_SNAPSHOT)
    print("Post-install PyTorch:", POST_INSTALL_TORCH_SNAPSHOT)

    if POST_INSTALL_TORCH_SNAPSHOT["torch_version"] != PRE_INSTALL_TORCH_SNAPSHOT["torch_version"]:
        raise RuntimeError(
            f"ABORT: pip altered PyTorch version ({PRE_INSTALL_TORCH_SNAPSHOT['torch_version']} -> "
            f"{POST_INSTALL_TORCH_SNAPSHOT['torch_version']}). Restart runtime before continuing."
        )
    if PRE_INSTALL_TORCH_SNAPSHOT["cuda_available"] and not POST_INSTALL_TORCH_SNAPSHOT["cuda_available"]:
        raise RuntimeError(
            "ABORT: CUDA availability was broken after dependency installation! "
            "Do not proceed; restart Colab runtime and inspect conflicting packages."
        )
    if PRE_INSTALL_TORCH_SNAPSHOT["torch_cuda_version"] != POST_INSTALL_TORCH_SNAPSHOT["torch_cuda_version"]:
        raise RuntimeError(
            f"ABORT: PyTorch CUDA build changed ({PRE_INSTALL_TORCH_SNAPSHOT['torch_cuda_version']} -> "
            f"{POST_INSTALL_TORCH_SNAPSHOT['torch_cuda_version']})."
        )

    # Capture exact installed versions (freeze snapshot)
    freeze_out = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True, text=True, check=True,
    ).stdout.splitlines()

    key_versions = {}
    for mod_name in [
        "torch", "torchaudio", "numpy", "scipy", "librosa", "soundfile",
        "transformers", "laion_clap", "torchlibrosa", "loralib", "einops", "psutil", "lightning",
    ]:
        m = importlib.import_module(mod_name)
        key_versions[mod_name] = getattr(m, "__version__", "installed")
        print(f"  {mod_name:14s}: {key_versions[mod_name]}")

    ENVIRONMENT_MANIFEST = {
        "python_version": platform.python_version(),
        "pinned_git_commits": pinned_heads,
        "torch_before_install": PRE_INSTALL_TORCH_SNAPSHOT,
        "torch_after_install": POST_INSTALL_TORCH_SNAPSHOT,
        "key_package_versions": key_versions,
        "pip_freeze": freeze_out,
        "reproducibility_note": (
            "Exact replay requires this captured pip_freeze snapshot + Colab OS/CUDA driver image; "
            "version ranges alone do not guarantee reproducibility."
        ),
    }
    (WORK_ROOT / "environment_manifest.json").write_text(
        json.dumps(ENVIRONMENT_MANIFEST, indent=2), encoding="utf-8"
    )
    HARDWARE_INFO["post_install_torch"] = POST_INSTALL_TORCH_SNAPSHOT
    print("\n[OK] Pinned repos verified, CUDA PyTorch intact, and environment_manifest.json saved.")

## Section 3 — Immutable Checkpoint Download, Integrity Verification & Scoped Safe Loader

**No checkpoints are downloaded until you explicitly execute this cell.**
- Downloads strictly from **pinned immutable Hugging Face Space revisions** (`revision=<commit_sha>`), NEVER from mutable `main`.
- Verifies **exact byte size** AND **exact expected SHA-256 hash** before registering each path in `VERIFIED_CHECKPOINT_PATHS`. **Aborts immediately** if any size or hash fails.
- **Honest Authentication Labels:** Because upstream authors do not publish independent out-of-band cryptographic signatures, `best_model.ckpt` and `audiosep_base_4M_steps.ckpt` are explicitly labeled `not_independently_authenticated`, and the two LAION-CLAP checkpoints are labeled `cross_repo_matched_same_platform_not_independently_signed`. Recording or checking same-host LFS hashes is transport/revision integrity checking, **not** cryptographic authentication.
- Defines `scoped_trusted_torch_load(...)`: attempts `weights_only=True` first; only if legacy pickle globals in a **pre-verified checkpoint path** require `weights_only=False` does it allow it strictly for that exact path, restoring `torch.load` in `finally:`.

In [ ]:
import contextlib
import hashlib
import json
import os
import shutil
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download

DOWNLOAD_CLAPSEP = True
DOWNLOAD_AUDIOSEP = True

CKPT_DIR = WORK_ROOT / "checkpoints"
CLAPSEP_CKPT_DIR = CKPT_DIR / "clapsep"
AUDIOSEP_CKPT_DIR = AUDIOSEP_REPO / "checkpoint"  # AudioSep's clap_encoder.py expects checkpoint/ under repo
CLAPSEP_CKPT_DIR.mkdir(parents=True, exist_ok=True)
AUDIOSEP_CKPT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_SPECS = []
if DOWNLOAD_CLAPSEP:
    CHECKPOINT_SPECS.extend([
        {
            "model": "CLAPSep",
            "repo_id": "AisakaMikoto/CLAPSep",
            "repo_type": "space",
            "revision": "e3c73653fa9aadcddbcc3b5e967182a573589e81",
            "filename": "model/best_model.ckpt",
            "expected_bytes": 177818986,
            "expected_sha256": "6fcc8dbcd7174af86266cf16b4105eced0802352762f81dbfefdce29af3dba04",
            "authentication_status": "not_independently_authenticated (same-host HF LFS OID only)",
            "license": "MIT (Aisaka0v0/CLAPSep)",
            "dest_path": CLAPSEP_CKPT_DIR / "best_model.ckpt",
        },
        {
            "model": "CLAPSep",
            "repo_id": "AisakaMikoto/CLAPSep",
            "repo_type": "space",
            "revision": "e3c73653fa9aadcddbcc3b5e967182a573589e81",
            "filename": "model/music_audioset_epoch_15_esc_90.14.pt",
            "expected_bytes": 2352471003,
            "expected_sha256": "fae3e9c087f2909c28a09dc31c8dfcdacbc42ba44c70e972b58c1bd1caf6dedd",
            "authentication_status": "cross_repo_matched_same_platform_not_independently_signed (matches lukewys/laion_clap@b3708341)",
            "license": "CC0-1.0 (LAION-CLAP)",
            "dest_path": CLAPSEP_CKPT_DIR / "music_audioset_epoch_15_esc_90.14.pt",
        },
    ])

if DOWNLOAD_AUDIOSEP:
    CHECKPOINT_SPECS.extend([
        {
            "model": "AudioSep",
            "repo_id": "Audio-AGI/AudioSep",
            "repo_type": "space",
            "revision": "0807a713283035d8b0d0628f1b8e3d6580459f7b",
            "filename": "checkpoint/audiosep_base_4M_steps.ckpt",
            "expected_bytes": 1264844076,
            "expected_sha256": "f8cda01bfd0ebd141eef45d41db7a3ada23a56568465840d3cff04b8010ce82c",
            "authentication_status": "not_independently_authenticated (same-host HF LFS OID only)",
            "license": "MIT (Audio-AGI/AudioSep)",
            "dest_path": AUDIOSEP_CKPT_DIR / "audiosep_base_4M_steps.ckpt",
        },
        {
            "model": "AudioSep",
            "repo_id": "Audio-AGI/AudioSep",
            "repo_type": "space",
            "revision": "0807a713283035d8b0d0628f1b8e3d6580459f7b",
            "filename": "checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt",
            "expected_bytes": 2352471003,
            "expected_sha256": "51c68f12f9d7ea25fdaaccf741ec7f81e93ee594455410f3bca4f47f88d8e006",
            "authentication_status": "cross_repo_matched_same_platform_not_independently_signed (matches lukewys/laion_clap@b3708341)",
            "license": "CC0-1.0 (LAION-CLAP)",
            "dest_path": AUDIOSEP_CKPT_DIR / "music_speech_audioset_epoch_15_esc_89.98.pt",
        },
    ])


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


VERIFIED_CHECKPOINT_PATHS: set[Path] = set()
CHECKPOINT_MANIFEST = []

for spec in CHECKPOINT_SPECS:
    dest: Path = spec["dest_path"]
    if not dest.exists() or dest.stat().st_size == 0:
        print(
            f"Downloading [{spec['model']}] {spec['repo_id']}@{spec['revision'][:10]} :: "
            f"{spec['filename']} ({spec['expected_bytes'] / (1024 ** 2):.2f} MB)..."
        )
        cached = hf_hub_download(
            repo_id=spec["repo_id"],
            repo_type=spec["repo_type"],
            revision=spec["revision"],
            filename=spec["filename"],
        )
        if dest.exists() or dest.is_symlink():
            dest.unlink()
        try:
            dest.symlink_to(Path(cached).resolve())
        except OSError:
            shutil.copy2(cached, dest)
    else:
        print(f"Already present: {dest} — running full byte-size & SHA-256 verification...")

    real_path = dest.resolve()
    actual_bytes = real_path.stat().st_size
    actual_sha256 = sha256_file(real_path)

    size_match = actual_bytes == spec["expected_bytes"]
    hash_match = actual_sha256.lower() == spec["expected_sha256"].lower()

    if not size_match or not hash_match:
        raise RuntimeError(
            f"ABORT: Checkpoint verification FAILED for {dest}!\n"
            f"  Bytes : got {actual_bytes:,}, expected {spec['expected_bytes']:,}\n"
            f"  SHA256: got {actual_sha256}, expected {spec['expected_sha256']}\n"
            "Refusing to deserialize unverified checkpoint."
        )

    VERIFIED_CHECKPOINT_PATHS.add(real_path)
    VERIFIED_CHECKPOINT_PATHS.add(dest.absolute())

    record = {
        "model": spec["model"],
        "source_url": f"https://huggingface.co/spaces/{spec['repo_id']}/blob/{spec['revision']}/{spec['filename']}",
        "pinned_revision": spec["revision"],
        "license": spec["license"],
        "authentication_status": spec["authentication_status"],
        "local_path": str(dest),
        "resolved_path": str(real_path),
        "actual_bytes": actual_bytes,
        "expected_bytes": spec["expected_bytes"],
        "actual_sha256": actual_sha256,
        "expected_sha256": spec["expected_sha256"],
        "integrity_verified": True,
    }
    CHECKPOINT_MANIFEST.append(record)
    print(f"  [VERIFIED] {actual_bytes:,} bytes | SHA-256 match | {spec['authentication_status']}")

(WORK_ROOT / "checkpoint_manifest.json").write_text(json.dumps(CHECKPOINT_MANIFEST, indent=2), encoding="utf-8")
print("\n[OK] All checkpoints verified against pinned revisions, exact byte counts, and expected SHA-256 digests.")


def load_checkpoint_safe_first(path: Path, map_location="cpu"):
    """Load a pre-verified checkpoint trying weights_only=True first, falling back only if legacy pickle globals require it."""
    resolved = path.resolve()
    if resolved not in VERIFIED_CHECKPOINT_PATHS and path.absolute() not in VERIFIED_CHECKPOINT_PATHS:
        raise PermissionError(f"Refusing to load unverified checkpoint path: {path}")
    try:
        obj = torch.load(str(path), map_location=map_location, weights_only=True)
        print(f"  [SAFE LOAD] {path.name} loaded with weights_only=True (no unrestricted pickle needed).")
        return obj, "weights_only=True"
    except Exception as exc:
        print(
            f"  [SCOPED FALLBACK] {path.name} rejected by weights_only=True ({type(exc).__name__}: {str(exc)[:140]}). "
            "Retrying with weights_only=False strictly for this SHA-256-verified checkpoint."
        )
        obj = torch.load(str(path), map_location=map_location, weights_only=False)
        return obj, f"weights_only=False (fallback after {type(exc).__name__})"


@contextlib.contextmanager
def scoped_trusted_torch_load(allowed_paths: set[Path], audit_log: list[dict]):
    """Temporarily intercept torch.load ONLY for explicitly verified checkpoint paths during legacy upstream loaders.

    - Attempts weights_only=True first even inside upstream library calls.
    - Falls back to weights_only=False ONLY if the target file resolves to an exact path in `allowed_paths`
      (which must already be in VERIFIED_CHECKPOINT_PATHS).
    - Raises PermissionError if an unverified or arbitrary path attempts fallback.
    - Always restores original torch.load on exit.
    """
    resolved_allowed = {p.resolve() for p in allowed_paths}
    for p in resolved_allowed:
        if p not in VERIFIED_CHECKPOINT_PATHS:
            raise PermissionError(f"Attempted to allowlist unverified path: {p}")

    orig_torch_load = torch.load

    def _guarded_torch_load(f, *args, **kwargs):
        raw_name = (
            f
            if isinstance(f, (str, os.PathLike))
            else (getattr(f, "name", None) or getattr(f, "path", None))
        )
        target_resolved = Path(raw_name).resolve() if isinstance(raw_name, (str, os.PathLike)) else None

        # Respect explicit non-None weights_only passed by caller
        if kwargs.get("weights_only") is not None:
            if kwargs["weights_only"] is False and (target_resolved is None or target_resolved not in resolved_allowed):
                raise PermissionError(
                    f"Blocked explicit weights_only=False for non-allowlisted target: {raw_name or f}"
                )
            return orig_torch_load(f, *args, **kwargs)

        clean_kwargs = {k: v for k, v in kwargs.items() if k != "weights_only"}
        start_pos = f.tell() if hasattr(f, "tell") and hasattr(f, "seek") else None

        # First try strict safe mode (weights_only=True)
        try:
            res = orig_torch_load(f, *args, weights_only=True, **clean_kwargs)
            audit_log.append({"file": str(target_resolved or raw_name or f), "mode": "weights_only=True"})
            return res
        except Exception as exc:
            if target_resolved is None or target_resolved not in resolved_allowed:
                raise PermissionError(
                    f"Blocked unrestricted pickle fallback for non-allowlisted target: {raw_name or f}"
                ) from exc
            if start_pos is not None:
                f.seek(start_pos)
            audit_log.append({
                "file": str(target_resolved),
                "mode": f"weights_only=False (scoped fallback after {type(exc).__name__})",
            })
            return orig_torch_load(f, *args, weights_only=False, **clean_kwargs)

    torch.load = _guarded_torch_load
    try:
        yield
    finally:
        torch.load = orig_torch_load


CHECKPOINT_LOAD_AUDIT_LOG: list[dict] = []
print("[OK] Scoped checkpoint loader initialized (no global TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD override).")

## Section 4 — Real Ukulele Recording Upload (Optional, Original Preserved Unchanged)

Upload your real ukulele recording (e.g., `.mp3`, `.wav`, or `.flac`) from your computer, or point `LOCAL_REAL_AUDIO_PATH` to an existing file in the Colab session.

**Strict Integrity Guarantees:**
- The uploaded file is saved **bit-for-bit unchanged** in `inputs/real_ukulele/` with SHA-256 verification and decoded ONLY via audio libraries (`librosa`/`soundfile`), never `torch.load`.
- Because the earlier local SONORA Stage 4 session files expired and are missing, **no Stage 4 reference output is fabricated or substituted**.
- If you do not have the real recording right now, set `UPLOAD_REAL_RECORDING = False`.

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

UPLOAD_REAL_RECORDING = True       # Set False to skip real-file upload
LOCAL_REAL_AUDIO_PATH = ""         # Optional: local Colab path if already present

REAL_INPUT_DIR = WORK_ROOT / "inputs" / "real_ukulele"
REAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
REAL_RECORDING_INFO = None


def _sha256_path(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


uploaded_path = None
if LOCAL_REAL_AUDIO_PATH.strip():
    src = Path(LOCAL_REAL_AUDIO_PATH.strip()).expanduser().resolve()
    if not src.is_file():
        raise FileNotFoundError(f"LOCAL_REAL_AUDIO_PATH not found: {src}")
    dst = REAL_INPUT_DIR / f"original_upload{src.suffix.lower()}"
    shutil.copy2(src, dst)
    uploaded_path = dst
elif UPLOAD_REAL_RECORDING:
    try:
        from google.colab import files
        print("Select your real ukulele recording (.mp3, .wav, or .flac) to upload:")
        uploaded = files.upload()
        if uploaded:
            orig_name, raw_bytes = next(iter(uploaded.items()))
            ext = Path(orig_name).suffix.lower() or ".bin"
            dst = REAL_INPUT_DIR / f"original_upload{ext}"
            dst.write_bytes(raw_bytes)  # Preserved bit-for-bit unchanged
            uploaded_path = dst
            print(f"Saved unchanged upload: {orig_name} -> {dst} ({len(raw_bytes):,} bytes)")
        else:
            print("No file uploaded. Real-recording track will be skipped.")
    except ImportError:
        print("Not running inside Google Colab UI. Set LOCAL_REAL_AUDIO_PATH to include a real recording.")
else:
    print("UPLOAD_REAL_RECORDING is False. Skipping real-recording track.")

if uploaded_path is not None:
    y_native, native_sr = librosa.load(str(uploaded_path), sr=None, mono=False)
    if y_native.ndim == 1:
        y_2d = y_native[:, np.newaxis].astype(np.float32)
    else:
        y_2d = y_native.T.astype(np.float32)

    native_wav_path = REAL_INPUT_DIR / "original_decoded_native_f32.wav"
    sf.write(str(native_wav_path), y_2d, int(native_sr), subtype="FLOAT", format="WAV")

    y_mono = np.mean(y_2d, axis=1, dtype=np.float64)
    y_32k = librosa.resample(y_mono, orig_sr=int(native_sr), target_sr=32000, res_type="soxr_hq").astype(np.float32)
    mono_32k_path = REAL_INPUT_DIR / "original_32k_mono_f32.wav"
    sf.write(str(mono_32k_path), y_32k, 32000, subtype="FLOAT", format="WAV")

    rms_dbfs = round(float(20.0 * np.log10(max(np.sqrt(np.mean(y_2d.astype(np.float64) ** 2)), 1e-12))), 2)
    peak_dbfs = round(float(20.0 * np.log10(max(np.max(np.abs(y_2d)), 1e-12))), 2)

    REAL_RECORDING_INFO = {
        "original_upload_path": str(uploaded_path),
        "original_upload_sha256": _sha256_path(uploaded_path),
        "original_upload_bytes": uploaded_path.stat().st_size,
        "decoded_native_wav_path": str(native_wav_path),
        "input_32k_mono_wav_path": str(mono_32k_path),
        "native_sample_rate": int(native_sr),
        "native_channels": int(y_2d.shape[1]),
        "native_frames": int(y_2d.shape[0]),
        "duration_seconds": round(float(y_2d.shape[0]) / float(native_sr), 4),
        "rms_dbfs": rms_dbfs,
        "sample_peak_dbfs": peak_dbfs,
        "stage4_reference_available": False,
        "stage4_reference_note": "Expired local SONORA Stage 4 session output is missing; NOT fabricated or substituted.",
    }
    (REAL_INPUT_DIR / "real_recording_manifest.json").write_text(
        json.dumps(REAL_RECORDING_INFO, indent=2), encoding="utf-8"
    )
    print("\n=== REAL RECORDING VERIFIED ===")
    for k, v in REAL_RECORDING_INFO.items():
        print(f"  {k:28s}: {v}")

## Section 5 — Calibrated Mixtures: Synthetic Interference (5A) & Optional Real Human Speech (5B)

Generates deterministic **48,000 Hz** reference mixtures with **separate clean ukulele target $s$ and isolated interference $n$** saved on disk at **multiple SNRs (`0 dB` and `+6 dB`)**.

### Scenarios Included
- **5A (Always generated — synthetic DSP proxies):**
  1. Clean ukulele (regression guard — no interference)
  2. Ukulele + synthetic speech proxy *(formant-modulated syllabic proxy — NOT real speech)*
  3. Ukulele + traffic *(road rumble + pass-by swells + distant horn)*
  4. Ukulele + fan noise *(airflow + blade-pass tonal hum)*
  5. Ukulele + wind *(gusting turbulence)*
  6. Ukulele + pink noise *($1/f$ broadband)*
  7. Ukulele + acoustic guitar *(adversarial plucked-string confusion test)*
- **5B (Optional user-supplied — real human speech interference):**
  - Allows uploading your own clean human speech WAV/FLAC/MP3 (never silently downloads copyrighted or unlicensed speech!).
  - Preserves the uploaded speech file bit-for-bit unchanged, decodes a separate 48 kHz mono working copy, calibrates mixtures at `0 dB` and `+6 dB SNR`, saves separate `clean`, `real_speech`, and `mixture` files, and tags every metric row with `interference_source="real_user_speech"` vs. `"synthetic_dsp_proxy"`.

### Explicit Limitations of Synthetic Signals vs. Real-World Recordings
- **Synthetic ukulele limitations:** Additive harmonic decay + bandpass finger clicks approximate re-entrant GCEA nylon strumming cleanly for repeatable DSP/AI measurements, but lack real wooden soundboard body modes, fret buzz, finger-sliding transients, natural player micro-timing, room early reflections, and microphone coloration. Models trained on real YouTube/AudioSet ukulele audio (CLAPSep/AudioSep) may score slightly differently on synthetic ukulele than on a real acoustic microphone recording!
- **Synthetic vs. real interference:** DSP-generated speech/traffic/fan/wind proxies lack real phonetics, natural room reverberation, and acoustic coupling with the instrument. **Use synthetic scores as controlled diagnostics, and rely on real-speech mixtures (5B) plus real ukulele recordings (Section 4) for final go/no-go decisions.**

In [ ]:
# === CELL 5A: SYNTHETIC CLEAN UKULELE & 6 SYNTHETIC INTERFERENCE SCENARIOS ===
import json
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy import signal

SYNTH_SR = 48000
SYNTH_DURATION_S = 8.0
SYNTH_SNR_DBS = [0.0, 6.0]
SYNTH_DIR = WORK_ROOT / "inputs" / "synthetic_benchmark"
SYNTH_DIR.mkdir(parents=True, exist_ok=True)


def _rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(x.astype(np.float64) ** 2)))


def scale_noise_to_snr(clean: np.ndarray, noise: np.ndarray, target_snr_db: float) -> np.ndarray:
    c_rms = _rms(clean)
    n_rms = _rms(noise)
    if c_rms <= 1e-12 or n_rms <= 1e-12:
        raise ValueError("Clean or noise signal has near-zero RMS.")
    desired_n_rms = c_rms / (10.0 ** (target_snr_db / 20.0))
    return (noise.astype(np.float64) * (desired_n_rms / n_rms)).astype(np.float64)


def synthesize_clean_ukulele(duration_s: float = SYNTH_DURATION_S, sr: int = SYNTH_SR, seed: int = 101) -> np.ndarray:
    """Synthesize re-entrant GCEA nylon-string ukulele strumming with clear inter-strum gap."""
    rng = np.random.default_rng(seed)
    n = int(round(duration_s * sr))
    out = np.zeros(n, dtype=np.float64)

    chords = [
        [392.00, 261.63, 329.63, 523.25],  # C major
        [392.00, 293.66, 392.00, 493.88],  # G major
        [440.00, 261.63, 329.63, 440.00],  # A minor
        [440.00, 261.63, 349.23, 440.00],  # F major
    ]
    strum_times = [0.25, 0.70, 1.15, 1.60, 2.05, 2.50, 2.95, 3.40, 5.25, 5.70, 6.15, 6.60, 7.05]

    for idx, t0 in enumerate(strum_times):
        chord = chords[(idx // 2) % len(chords)]
        ordered_freqs = chord if (idx % 2 == 0) else list(reversed(chord))
        for s_idx, f0 in enumerate(ordered_freqs):
            start = int(round((t0 + s_idx * 0.011) * sr))
            note_len = min(int(round(0.75 * sr)), n - start)
            if note_len <= 16:
                continue
            t = np.arange(note_len, dtype=np.float64) / sr
            attack = 1.0 - np.exp(-t / 0.0022)
            note = np.zeros(note_len, dtype=np.float64)
            for k in range(1, 10):
                fk = f0 * k * (1.0 + 0.00018 * (k ** 2))
                if fk >= 0.45 * sr:
                    break
                amp = (1.0 / (k ** 1.35)) * (1.15 if k in (1, 2, 3) else 0.75)
                tau = max(0.055, 0.36 / (k ** 0.75))
                note += amp * np.sin(2.0 * np.pi * fk * t + rng.uniform(0.0, 2.0 * np.pi)) * np.exp(-t / tau)

            click_len = min(int(0.014 * sr), note_len)
            click = rng.normal(0.0, 1.0, size=click_len) * np.exp(-np.arange(click_len) / (0.0025 * sr))
            sos_click = signal.butter(2, [1100.0, 5500.0], btype="bandpass", fs=sr, output="sos")
            note[:click_len] += signal.sosfilt(sos_click, click) * 0.22
            out[start : start + note_len] += attack * note * 0.24

    sos_body = signal.butter(2, 180.0, btype="highpass", fs=sr, output="sos")
    out = signal.sosfiltfilt(sos_body, out)
    out = out * (0.45 / (np.max(np.abs(out)) + 1e-12))
    return out.astype(np.float64)


def synthesize_speech_proxy(duration_s: float = SYNTH_DURATION_S, sr: int = SYNTH_SR, seed: int = 201) -> np.ndarray:
    """Formant-modulated syllabic speech proxy (NOT real human speech)."""
    rng = np.random.default_rng(seed)
    n = int(round(duration_s * sr))
    t = np.arange(n, dtype=np.float64) / sr
    f0 = 145.0 + 30.0 * np.sin(2.0 * np.pi * 0.85 * t) + 12.0 * np.sin(2.0 * np.pi * 2.1 * t)
    phase = 2.0 * np.pi * np.cumsum(f0) / sr
    glottal = sum((1.0 / (k ** 1.1)) * np.sin(k * phase) for k in range(1, 18))
    syllable_env = np.clip(0.55 + 0.55 * np.sin(2.0 * np.pi * 3.8 * t + 0.4) * np.sin(2.0 * np.pi * 0.65 * t), 0.0, 1.0)
    y = np.zeros(n, dtype=np.float64)
    for fc, bw, gain in [(520.0, 140.0, 1.0), (1480.0, 260.0, 0.75), (2650.0, 360.0, 0.40)]:
        sos = signal.butter(2, [max(80.0, fc - bw / 2), min(0.45 * sr, fc + bw / 2)], btype="bandpass", fs=sr, output="sos")
        y += gain * signal.sosfilt(sos, glottal)
    fric = rng.normal(0.0, 1.0, size=n)
    sos_f = signal.butter(2, [3500.0, 7500.0], btype="bandpass", fs=sr, output="sos")
    fric_env = np.clip(np.sin(2.0 * np.pi * 2.3 * t + 1.1) - 0.65, 0.0, 1.0) * 2.5
    y = syllable_env * y + 0.18 * fric_env * signal.sosfilt(sos_f, fric)
    return y / (np.max(np.abs(y)) + 1e-12)


def synthesize_traffic_noise(duration_s: float = SYNTH_DURATION_S, sr: int = SYNTH_SR, seed: int = 202) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = int(round(duration_s * sr))
    t = np.arange(n, dtype=np.float64) / sr
    road = signal.sosfilt(signal.butter(3, [40.0, 1200.0], btype="bandpass", fs=sr, output="sos"), rng.normal(0.0, 1.0, size=n))
    passby = 0.45 + 0.85 * np.exp(-0.5 * ((t - 2.3) / 0.65) ** 2) + 0.95 * np.exp(-0.5 * ((t - 5.9) / 0.75) ** 2)
    engine = 0.25 * np.sin(2.0 * np.pi * (82.0 + 14.0 * np.exp(-0.5 * ((t - 2.3) / 0.6) ** 2)) * t) + 0.15 * np.sin(2.0 * np.pi * 164.0 * t)
    horn_env = np.exp(-0.5 * ((t - 4.6) / 0.16) ** 6)
    horn = 0.22 * horn_env * (np.sin(2.0 * np.pi * 415.0 * t) + 0.85 * np.sin(2.0 * np.pi * 523.0 * t))
    y = passby * (road + engine) + horn
    return y / (np.max(np.abs(y)) + 1e-12)


def synthesize_fan_noise(duration_s: float = SYNTH_DURATION_S, sr: int = SYNTH_SR, seed: int = 203) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = int(round(duration_s * sr))
    t = np.arange(n, dtype=np.float64) / sr
    airflow = signal.sosfilt(signal.butter(2, [90.0, 4800.0], btype="bandpass", fs=sr, output="sos"), rng.normal(0.0, 1.0, size=n))
    airflow = airflow * (1.0 + 0.05 * np.sin(2.0 * np.pi * 1.4 * t))
    motor = 0.28 * np.sin(2.0 * np.pi * 100.0 * t) + 0.16 * np.sin(2.0 * np.pi * 200.0 * t) + 0.08 * np.sin(2.0 * np.pi * 300.0 * t)
    y = airflow + motor
    return y / (np.max(np.abs(y)) + 1e-12)


def synthesize_wind_noise(duration_s: float = SYNTH_DURATION_S, sr: int = SYNTH_SR, seed: int = 204) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = int(round(duration_s * sr))
    t = np.arange(n, dtype=np.float64) / sr
    white = rng.normal(0.0, 1.0, size=n)
    gust_env = 0.30 + 0.70 * np.clip(0.5 + 0.4 * np.sin(2.0 * np.pi * 0.37 * t) + 0.35 * np.sin(2.0 * np.pi * 0.91 * t + 1.2), 0.0, 1.0)
    y = gust_env * (
        1.2 * signal.sosfilt(signal.butter(3, 380.0, btype="lowpass", fs=sr, output="sos"), white)
        + 0.35 * signal.sosfilt(signal.butter(2, [350.0, 2200.0], btype="bandpass", fs=sr, output="sos"), white)
    )
    return y / (np.max(np.abs(y)) + 1e-12)


def synthesize_pink_noise(duration_s: float = SYNTH_DURATION_S, sr: int = SYNTH_SR, seed: int = 205) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = int(round(duration_s * sr))
    spec = np.fft.rfft(rng.normal(0.0, 1.0, size=n))
    freqs = np.fft.rfftfreq(n, d=1.0 / sr)
    scale = 1.0 / np.sqrt(np.maximum(freqs, 20.0))
    scale[0] = 0.0
    pink = np.fft.irfft(spec * scale, n=n)
    return pink / (np.max(np.abs(pink)) + 1e-12)


def synthesize_acoustic_guitar(duration_s: float = SYNTH_DURATION_S, sr: int = SYNTH_SR, seed: int = 206) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = int(round(duration_s * sr))
    out = np.zeros(n, dtype=np.float64)
    chords = [
        [110.00, 164.81, 220.00, 261.63, 329.63],
        [98.00, 123.47, 146.83, 196.00, 246.94, 392.00],
        [146.83, 220.00, 293.66, 369.99],
        [82.41, 123.47, 164.81, 196.00, 246.94, 329.63],
    ]
    for idx, t0 in enumerate([0.10, 0.90, 1.70, 2.50, 3.30, 4.10, 4.90, 5.70, 6.50, 7.30]):
        for s_idx, f0 in enumerate(chords[idx % len(chords)]):
            start = int(round((t0 + s_idx * 0.015) * sr))
            note_len = min(int(round(1.05 * sr)), n - start)
            if note_len <= 16:
                continue
            t = np.arange(note_len, dtype=np.float64) / sr
            attack = 1.0 - np.exp(-t / 0.0018)
            note = np.zeros(note_len, dtype=np.float64)
            for k in range(1, 15):
                fk = f0 * k * (1.0 + 0.00012 * (k ** 2))
                if fk >= 0.45 * sr:
                    break
                note += (1.0 / (k ** 1.1)) * np.sin(2.0 * np.pi * fk * t + rng.uniform(0, 2 * np.pi)) * np.exp(-t / max(0.08, 0.55 / (k ** 0.65)))
            out[start : start + note_len] += attack * note * 0.16
    return out / (np.max(np.abs(out)) + 1e-12)


clean_uke = synthesize_clean_ukulele()
clean_path = SYNTH_DIR / "clean_ukulele_48k.wav"
sf.write(str(clean_path), clean_uke.astype(np.float32), SYNTH_SR, subtype="FLOAT")

GAP_RANGE_SAMPLES = [int(4.15 * SYNTH_SR), int(5.15 * SYNTH_SR)]

NOISE_GENERATORS = {
    "speech_proxy": ("Synthetic Speech Proxy (formant DSP)", synthesize_speech_proxy),
    "traffic": ("Traffic (road rumble + pass-bys + horn)", synthesize_traffic_noise),
    "fan": ("Fan Noise (airflow + blade hum)", synthesize_fan_noise),
    "wind": ("Wind (gusting turbulence)", synthesize_wind_noise),
    "pink_noise": ("Pink Noise (1/f broadband)", synthesize_pink_noise),
    "acoustic_guitar": ("Acoustic Guitar (adversarial plucked string)", synthesize_acoustic_guitar),
}

SYNTHETIC_SCENARIOS = [
    {
        "scenario_id": "clean_ukulele",
        "noise_type": "none",
        "interference_source": "none",
        "label": "Clean Ukulele (regression guard)",
        "target_snr_db": None,
        "clean_path": str(clean_path),
        "noise_path": None,
        "mixture_path": str(clean_path),
        "sample_rate": SYNTH_SR,
        "gap_range_samples": GAP_RANGE_SAMPLES,
    }
]


def add_calibrated_mixture_scenarios(
    noise_key: str,
    label: str,
    interference_source: str,
    raw_noise: np.ndarray,
    snr_list: list[float] = SYNTH_SNR_DBS,
) -> None:
    for snr_db in snr_list:
        scaled_noise = scale_noise_to_snr(clean_uke, raw_noise, snr_db)
        mixture = clean_uke + scaled_noise
        peak = float(np.max(np.abs(mixture)))
        scale = (0.92 / peak) if peak > 0.92 else 1.0
        c_s = (clean_uke * scale).astype(np.float32)
        n_s = (scaled_noise * scale).astype(np.float32)
        m_s = (mixture * scale).astype(np.float32)

        sid = f"ukulele_plus_{noise_key}_snr{int(snr_db)}db"
        c_p = SYNTH_DIR / f"{sid}__clean.wav"
        n_p = SYNTH_DIR / f"{sid}__noise.wav"
        m_p = SYNTH_DIR / f"{sid}__mixture.wav"
        sf.write(str(c_p), c_s, SYNTH_SR, subtype="FLOAT")
        sf.write(str(n_p), n_s, SYNTH_SR, subtype="FLOAT")
        sf.write(str(m_p), m_s, SYNTH_SR, subtype="FLOAT")

        SYNTHETIC_SCENARIOS.append(
            {
                "scenario_id": sid,
                "noise_type": noise_key,
                "interference_source": interference_source,
                "label": f"Ukulele + {label} ({snr_db:+.0f} dB SNR)",
                "target_snr_db": snr_db,
                "clean_path": str(c_p),
                "noise_path": str(n_p),
                "mixture_path": str(m_p),
                "sample_rate": SYNTH_SR,
                "gap_range_samples": GAP_RANGE_SAMPLES,
            }
        )


for n_key, (n_label, gen_fn) in NOISE_GENERATORS.items():
    add_calibrated_mixture_scenarios(n_key, n_label, "synthetic_dsp_proxy", gen_fn())

print(f"[OK] Generated {len(SYNTHETIC_SCENARIOS)} synthetic scenarios at {SYNTH_SR} Hz.")

In [ ]:
# === CELL 5B: OPTIONAL REAL HUMAN SPEECH INTERFERENCE MIXTURE (0 dB & +6 dB SNR) ===
# Never silently downloads copyrighted or unlicensed speech.
# Supply your own clean human speech WAV/FLAC/MP3 via upload or LOCAL_REAL_SPEECH_PATH.
import json
import shutil
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

UPLOAD_REAL_SPEECH = False        # Set True to upload a real clean speech recording from your computer
LOCAL_REAL_SPEECH_PATH = ""       # Optional: existing local Colab path to a real speech file

REAL_SPEECH_DIR = WORK_ROOT / "inputs" / "real_speech"
REAL_SPEECH_DIR.mkdir(parents=True, exist_ok=True)
REAL_SPEECH_INFO = None

speech_src_saved = None
if LOCAL_REAL_SPEECH_PATH.strip():
    src = Path(LOCAL_REAL_SPEECH_PATH.strip()).expanduser().resolve()
    if not src.is_file():
        raise FileNotFoundError(f"LOCAL_REAL_SPEECH_PATH not found: {src}")
    dst = REAL_SPEECH_DIR / f"original_real_speech{src.suffix.lower()}"
    shutil.copy2(src, dst)  # Preserved bit-for-bit unchanged
    speech_src_saved = dst
elif UPLOAD_REAL_SPEECH:
    try:
        from google.colab import files
        print("Select your clean real human speech recording (.wav, .flac, or .mp3):")
        uploaded = files.upload()
        if uploaded:
            orig_name, raw_bytes = next(iter(uploaded.items()))
            ext = Path(orig_name).suffix.lower() or ".bin"
            dst = REAL_SPEECH_DIR / f"original_real_speech{ext}"
            dst.write_bytes(raw_bytes)  # Preserved bit-for-bit unchanged
            speech_src_saved = dst
            print(f"Saved unchanged real speech upload: {orig_name} -> {dst} ({len(raw_bytes):,} bytes)")
    except ImportError:
        print("Not running in Google Colab UI. Set LOCAL_REAL_SPEECH_PATH to add real speech interference.")

if speech_src_saved is not None:
    y_sp, sr_sp = librosa.load(str(speech_src_saved), sr=None, mono=True)
    y_sp_48k = librosa.resample(y_sp.astype(np.float64), orig_sr=int(sr_sp), target_sr=SYNTH_SR, res_type="soxr_hq")
    # Save separate decoded working copy before duration alignment
    decoded_sp_path = REAL_SPEECH_DIR / "decoded_real_speech_48k_mono_f32.wav"
    sf.write(str(decoded_sp_path), y_sp_48k.astype(np.float32), SYNTH_SR, subtype="FLOAT")

    target_n = len(clean_uke)
    if len(y_sp_48k) >= target_n:
        speech_aligned = y_sp_48k[:target_n]
        align_action = f"trimmed from {len(y_sp_48k) / SYNTH_SR:.2f}s to {target_n / SYNTH_SR:.2f}s"
    else:
        reps = int(np.ceil(target_n / max(len(y_sp_48k), 1)))
        speech_aligned = np.tile(y_sp_48k, reps)[:target_n]
        align_action = f"looped from {len(y_sp_48k) / SYNTH_SR:.2f}s to {target_n / SYNTH_SR:.2f}s"

    # Remove any DC offset and guard against silence
    speech_aligned = speech_aligned - np.mean(speech_aligned)
    if _rms(speech_aligned) < 1e-6:
        raise ValueError("Uploaded real speech audio is silent or near-silent; cannot calibrate SNR.")

    add_calibrated_mixture_scenarios(
        noise_key="real_speech",
        label="Real Human Speech (user-supplied)",
        interference_source="real_user_speech",
        raw_noise=speech_aligned,
        snr_list=SYNTH_SNR_DBS,
    )
    REAL_SPEECH_INFO = {
        "original_upload_path": str(speech_src_saved),
        "original_upload_sha256": _sha256_path(speech_src_saved),
        "decoded_working_copy_path": str(decoded_sp_path),
        "original_sr": int(sr_sp),
        "alignment_action": align_action,
        "snr_levels_db": SYNTH_SNR_DBS,
    }
    (REAL_SPEECH_DIR / "real_speech_manifest.json").write_text(json.dumps(REAL_SPEECH_INFO, indent=2), encoding="utf-8")
    print(f"[OK] Added calibrated REAL human speech interference mixtures at {SYNTH_SNR_DBS} dB SNR ({align_action}).")
else:
    print("No real speech interference file supplied — only synthetic speech proxy will run (clearly labeled as synthetic_dsp_proxy).")

# Persist complete manifest including optional real speech scenarios
(SYNTH_DIR / "synthetic_manifest.json").write_text(json.dumps(SYNTHETIC_SCENARIOS, indent=2), encoding="utf-8")
print(f"Total calibrated reference scenarios ready: {len(SYNTHETIC_SCENARIOS)}")
for sc in SYNTHETIC_SCENARIOS:
    print(f"  - {sc['scenario_id']:38s} | [{sc['interference_source']:19s}] | {sc['label']}")

## Section 6 — Genuine Peak Memory Monitor, Waveform Finalization & Objective Metrics

### Key Engineering Safeguards Implemented Here
1. **Genuine Peak Memory Measurement (`PeakMemoryMonitor`):** Polls process RSS every 20 ms on a dedicated thread during model load and inference, captures Linux `/proc/self/status` `VmHWM` where available, and separates:
   - `ram_before_load_mb`, `ram_after_load_mb`, `peak_ram_during_load_mb`, `peak_ram_during_inference_mb`
   - `gpu_peak_allocated_mb` (`torch.cuda.max_memory_allocated`) and `gpu_peak_reserved_mb` (`torch.cuda.max_memory_reserved`)
2. **Exact Audio Output & Metric Consistency (`finalize_audition_waveform`):**
   - Preserves the raw unmodified `float32` WAV (`*_raw_f32.wav`, `subtype="FLOAT"`) for archival inspection.
   - Creates the exact audition waveform (`*_audition.wav`): **if** no sample exceeds `[-1.0, 1.0]`, the audition waveform is **bit-identical** to the raw float32 waveform (`was_clipped=False`). If samples exceed `[-1.0, 1.0]`, clips explicitly to `[-1.0, 1.0]`, records exact `raw_peak_linear`, `clipped_samples`, and `clipped_fraction`, and computes **ALL objective metrics and residuals strictly on that exact auditioned waveform**!
3. **Complete Implemented Metric Set (Aligned with SONORA Stage 4 `benchmarks/metrics.py`):**
   - Joint SNR (dB) & $\Delta\text{SNR}$, Joint SI-SDR (dB) & $\Delta\text{SI-SDR}$ — evaluated at **both 32 kHz** (pure separation) and **48 kHz native SR** (including the 16 kHz bandwidth cap).
   - **Active-Frame Spectral Correlation** (`active_frame_spectral_corr`): mean Pearson correlation of log-magnitude spectra on ukulele-active frames.
   - **Target Level Retention** (`target_level_retention_db`): RMS difference (`separated_active_rms_dbfs - clean_active_rms_dbfs`) on ukulele-active frames (`0.0 dB` = exact level preserved).
   - **Multi-Resolution Log-Spectral Distance (MR-LSD, dB):** averaged across `n_fft ∈ {512, 1024, 2048}`.
   - **Strumming Attack / Onset Preservation:** spectral-flux onset Pearson correlation (`n_fft=512`, `hop=128`) + crest-factor delta.
   - **Harmonic Preservation:** ukulele harmonic-band (`250–6000 Hz`) cosine similarity + spectral centroid shift (Hz).
   - **Background Noise Suppression (dB):** known-noise orthogonal projection attenuation AND inter-strum silent-gap RMS drop.
   - **Processing Difference / Residual:** `input_aligned[n] - audition_output[n]` (unnormalized; diagnostic difference, **never** claimed to be pure noise).

In [ ]:
import math
import os
import threading
import time
from pathlib import Path
from typing import Any

import librosa
import numpy as np
import psutil
import soundfile as sf
import torch
from scipy import signal

EPS = 1e-12
EXACT_MATCH_POWER_EPS = 1e-14
EXACT_MATCH_SENTINEL_DB = 120.0


def current_rss_mb() -> float:
    return round(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2), 1)


def process_vmhwm_mb() -> float | None:
    """Linux kernel-tracked process high-water mark RSS (VmHWM in MB) if available."""
    status_path = Path("/proc/self/status")
    if status_path.is_file():
        for line in status_path.read_text(encoding="utf-8", errors="ignore").splitlines():
            if line.startswith("VmHWM:"):
                parts = line.split()
                if len(parts) >= 2:
                    try:
                        return round(int(parts[1]) / 1024.0, 1)
                    except ValueError:
                        return None
    return None


class PeakMemoryMonitor:
    """Samples process RSS in a background thread to measure genuine peak RAM during an operation."""

    def __init__(self, interval_s: float = 0.02):
        self.interval_s = interval_s
        self._proc = psutil.Process(os.getpid())
        self._stop_evt = threading.Event()
        self._thread: threading.Thread | None = None
        self.ram_before_mb: float = 0.0
        self.ram_after_mb: float = 0.0
        self.peak_ram_mb: float = 0.0
        self.gpu_peak_allocated_mb: float | None = None
        self.gpu_peak_reserved_mb: float | None = None

    def _poll(self):
        while not self._stop_evt.is_set():
            rss = self._proc.memory_info().rss / (1024 ** 2)
            if rss > self.peak_ram_mb:
                self.peak_ram_mb = rss
            self._stop_evt.wait(self.interval_s)

    def __enter__(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        self.ram_before_mb = self._proc.memory_info().rss / (1024 ** 2)
        self.peak_ram_mb = self.ram_before_mb
        self._stop_evt.clear()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            self.gpu_peak_allocated_mb = round(torch.cuda.max_memory_allocated() / (1024 ** 2), 1)
            self.gpu_peak_reserved_mb = round(torch.cuda.max_memory_reserved() / (1024 ** 2), 1)
        rss_end = self._proc.memory_info().rss / (1024 ** 2)
        if rss_end > self.peak_ram_mb:
            self.peak_ram_mb = rss_end
        self._stop_evt.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.ram_before_mb = round(self.ram_before_mb, 1)
        self.ram_after_mb = round(rss_end, 1)
        self.peak_ram_mb = round(self.peak_ram_mb, 1)
        return False


def align_length(arr: np.ndarray, target_len: int) -> np.ndarray:
    a = np.asarray(arr, dtype=np.float64).reshape(-1)
    if len(a) == target_len:
        return a
    if len(a) > target_len:
        return a[:target_len]
    return np.pad(a, (0, target_len - len(a)))


def finalize_audition_waveform(raw_f32: np.ndarray) -> tuple[np.ndarray, dict[str, Any]]:
    """Produce the exact waveform used for BOTH saving/auditioning AND objective metric evaluation.

    - Preserves full precision within [-1.0, 1.0] with zero scaling.
    - Clips strictly to [-1.0, 1.0] ONLY if samples exceed full scale (matching WAV/browser playback)
      and records exact clipping telemetry so evaluated audio and auditioned audio are 100% identical.
    """
    raw = np.asarray(raw_f32, dtype=np.float32).reshape(-1)
    raw_peak = float(np.max(np.abs(raw))) if raw.size else 0.0
    over_mask = np.abs(raw) > 1.0
    clipped_samples = int(np.sum(over_mask))
    was_clipped = clipped_samples > 0
    audition = np.clip(raw, -1.0, 1.0).astype(np.float32) if was_clipped else raw.copy()
    info = {
        "raw_peak_linear": round(raw_peak, 6),
        "was_clipped": was_clipped,
        "clipped_samples": clipped_samples,
        "clipped_fraction": round(clipped_samples / max(len(raw), 1), 6),
        "gain_adjustment_db": 0.0,
    }
    return audition, info


def save_audio_artifacts(
    raw_f32: np.ndarray,
    audition_f32: np.ndarray,
    residual_f32: np.ndarray,
    sr: int,
    base_path_no_ext: Path,
) -> dict[str, str]:
    """Save unmodified float32 archive WAV + exact audition float32 WAV + exact aligned difference residual WAV."""
    raw_path = base_path_no_ext.with_name(base_path_no_ext.name + "__raw_f32.wav")
    audition_path = base_path_no_ext.with_name(base_path_no_ext.name + "__audition.wav")
    residual_path = base_path_no_ext.with_name(base_path_no_ext.name + "__diff_residual.wav")
    # Save 32-bit IEEE FLOAT WAVs so evaluated arrays and saved files are bit-identical (zero PCM quantization delta)
    sf.write(str(raw_path), np.asarray(raw_f32, dtype=np.float32), sr, subtype="FLOAT", format="WAV")
    sf.write(str(audition_path), np.asarray(audition_f32, dtype=np.float32), sr, subtype="FLOAT", format="WAV")
    sf.write(str(residual_path), np.asarray(residual_f32, dtype=np.float32), sr, subtype="FLOAT", format="WAV")
    return {
        "raw_f32_wav": str(raw_path),
        "audition_wav": str(audition_path),
        "residual_diff_wav": str(residual_path),
    }


def compute_snr_db(clean: np.ndarray, test: np.ndarray) -> float | None:
    c = np.asarray(clean, dtype=np.float64).reshape(-1)
    t = align_length(test, len(c))
    err = t - c
    sig_p = float(np.mean(c * c))
    err_p = float(np.mean(err * err))
    if sig_p <= EPS:
        return None
    if err_p <= EXACT_MATCH_POWER_EPS:
        return EXACT_MATCH_SENTINEL_DB
    return round(10.0 * math.log10(sig_p / err_p), 2)


def compute_si_sdr_db(clean: np.ndarray, estimate: np.ndarray) -> float | None:
    c = np.asarray(clean, dtype=np.float64).reshape(-1)
    e = align_length(estimate, len(c))
    s = c - np.mean(c)
    x = e - np.mean(e)
    s_energy = float(np.dot(s, s))
    if s_energy <= EPS:
        return None
    alpha = float(np.dot(x, s)) / s_energy
    s_target = alpha * s
    e_noise = x - s_target
    t_p = float(np.dot(s_target, s_target))
    n_p = float(np.dot(e_noise, e_noise))
    if n_p <= EXACT_MATCH_POWER_EPS:
        return EXACT_MATCH_SENTINEL_DB
    if t_p <= EPS:
        return -120.0
    return round(10.0 * math.log10(t_p / n_p), 2)


def compute_multi_res_lsd_db(
    clean: np.ndarray,
    estimate: np.ndarray,
    sample_rate: int,
    n_ffts: tuple[int, ...] = (512, 1024, 2048),
) -> float:
    c = np.asarray(clean, dtype=np.float64).reshape(-1)
    e = align_length(estimate, len(c))
    vals = []
    for n_fft in n_ffts:
        nf = min(n_fft, len(c))
        hop = max(64, nf // 4)
        _, _, zc = signal.stft(c, fs=sample_rate, nperseg=nf, noverlap=nf - hop)
        _, _, ze = signal.stft(e, fs=sample_rate, nperseg=nf, noverlap=nf - hop)
        pc = np.maximum(np.abs(zc) ** 2, 1e-10)
        pe = np.maximum(np.abs(ze) ** 2, 1e-10)
        diff_db = 10.0 * np.log10(pc) - 10.0 * np.log10(pe)
        vals.append(float(np.mean(np.sqrt(np.mean(diff_db ** 2, axis=0)))))
    return round(float(np.mean(vals)), 3)


def compute_onset_preservation(clean: np.ndarray, estimate: np.ndarray, sample_rate: int) -> float | None:
    c = np.asarray(clean, dtype=np.float64).reshape(-1)
    e = align_length(estimate, len(c))
    nf = min(512, len(c))
    hop = max(32, nf // 4)
    _, _, zc = signal.stft(c, fs=sample_rate, nperseg=nf, noverlap=nf - hop)
    _, _, ze = signal.stft(e, fs=sample_rate, nperseg=nf, noverlap=nf - hop)
    mc, me = np.abs(zc), np.abs(ze)
    if mc.shape[1] < 3:
        return None
    fc = np.sum(np.maximum(mc[:, 1:] - mc[:, :-1], 0.0), axis=0)
    fe = np.sum(np.maximum(me[:, 1:] - me[:, :-1], 0.0), axis=0)
    if float(np.std(fc)) < 1e-9 or float(np.std(fe)) < 1e-9:
        return None
    corr = float(np.corrcoef(fc, fe)[0, 1])
    return round(float(np.clip(corr, -1.0, 1.0)), 4) if math.isfinite(corr) else None


def compute_active_target_and_harmonic_metrics(
    clean: np.ndarray, estimate: np.ndarray, sample_rate: int
) -> dict[str, float | None]:
    """Compute active-frame spectral correlation, target level retention (dB), and 250-6000 Hz harmonic preservation."""
    c = np.asarray(clean, dtype=np.float64).reshape(-1)
    e = align_length(estimate, len(c))
    nf = min(2048, len(c))
    hop = nf // 4
    freqs, _, zc = signal.stft(c, fs=sample_rate, nperseg=nf, noverlap=nf - hop)
    _, _, ze = signal.stft(e, fs=sample_rate, nperseg=nf, noverlap=nf - hop)

    mag_c = np.abs(zc)
    mag_e = np.abs(ze)
    frame_energy = np.sum(mag_c ** 2, axis=0)
    active = frame_energy > (0.05 * float(np.max(frame_energy) + EPS))
    if not np.any(active):
        return {
            "active_frame_spectral_corr": None,
            "target_level_retention_db": None,
            "harmonic_cosine_sim": None,
            "harmonic_centroid_shift_hz": None,
        }

    # 1) Active-frame log-magnitude spectral Pearson correlation (200 Hz to Nyquist)
    full_band = freqs >= 200.0
    lc = np.log10(np.maximum(mag_c[full_band][:, active], 1e-6))
    le = np.log10(np.maximum(mag_e[full_band][:, active], 1e-6))
    frame_corrs = []
    for i in range(lc.shape[1]):
        if float(np.std(lc[:, i])) > 1e-9 and float(np.std(le[:, i])) > 1e-9:
            r = float(np.corrcoef(lc[:, i], le[:, i])[0, 1])
            if math.isfinite(r):
                frame_corrs.append(r)
    active_spec_corr = round(float(np.mean(frame_corrs)), 4) if frame_corrs else None

    # 2) Target level retention on active frames (dB; 0 dB = exact level retention)
    active_c_rms = math.sqrt(float(np.mean(np.sum(mag_c[:, active] ** 2, axis=0))) + EPS)
    active_e_rms = math.sqrt(float(np.mean(np.sum(mag_e[:, active] ** 2, axis=0))) + EPS)
    level_retention_db = round(20.0 * math.log10(max(active_e_rms, EPS) / max(active_c_rms, EPS)), 2)

    # 3) Harmonic-band (250-6000 Hz) cosine similarity & centroid shift
    hband = (freqs >= 250.0) & (freqs <= min(6000.0, 0.48 * sample_rate))
    mc_a = mag_c[hband][:, active]
    me_a = mag_e[hband][:, active]
    num = np.sum(mc_a * me_a, axis=0)
    den = np.sqrt(np.sum(mc_a ** 2, axis=0) * np.sum(me_a ** 2, axis=0)) + EPS
    cos_sim = round(float(np.mean(num / den)), 4)

    fb = freqs[hband][:, np.newaxis]
    c_cent = float(np.mean(np.sum(fb * mc_a, axis=0) / (np.sum(mc_a, axis=0) + EPS)))
    e_cent = float(np.mean(np.sum(fb * me_a, axis=0) / (np.sum(me_a, axis=0) + EPS)))

    return {
        "active_frame_spectral_corr": active_spec_corr,
        "target_level_retention_db": level_retention_db,
        "harmonic_cosine_sim": cos_sim,
        "harmonic_centroid_shift_hz": round(e_cent - c_cent, 1),
    }


def compute_background_suppression(
    clean: np.ndarray,
    noise: np.ndarray | None,
    mixture: np.ndarray,
    estimate: np.ndarray,
    gap_range_samples: list[int] | None = None,
) -> dict[str, float | None]:
    m = np.asarray(mixture, dtype=np.float64).reshape(-1)
    e = align_length(estimate, len(m))
    out: dict[str, float | None] = {
        "noise_projection_suppression_db": None,
        "gap_noise_suppression_db": None,
    }
    if noise is not None:
        n = align_length(noise, len(m))
        c = align_length(clean, len(m))
        c_energy = float(np.dot(c, c)) + EPS
        n_orth = n - (float(np.dot(n, c)) / c_energy) * c
        n_orth_p = float(np.dot(n_orth, n_orth))
        if n_orth_p > EPS:
            in_proj = abs(float(np.dot(m, n_orth))) / math.sqrt(n_orth_p)
            out_proj = abs(float(np.dot(e, n_orth))) / math.sqrt(n_orth_p)
            if in_proj > EPS:
                out["noise_projection_suppression_db"] = round(
                    20.0 * math.log10(max(in_proj, EPS) / max(out_proj, EPS)), 2
                )
    if gap_range_samples is not None and len(gap_range_samples) == 2:
        g0 = max(0, min(gap_range_samples[0], len(m) - 1))
        g1 = max(g0 + 1, min(gap_range_samples[1], len(m)))
        in_gap_rms = float(np.sqrt(np.mean(m[g0:g1] ** 2)))
        out_gap_rms = float(np.sqrt(np.mean(e[g0:g1] ** 2)))
        if in_gap_rms > 1e-7:
            out["gap_noise_suppression_db"] = round(
                20.0 * math.log10(max(in_gap_rms, EPS) / max(out_gap_rms, EPS)), 2
            )
    return out


def compute_crest_factor_db(x: np.ndarray) -> float | None:
    a = np.asarray(x, dtype=np.float64).reshape(-1)
    r = float(np.sqrt(np.mean(a * a)))
    p = float(np.max(np.abs(a)))
    if r <= EPS or p <= EPS:
        return None
    return round(20.0 * math.log10(p / r), 2)


def evaluate_calibrated_result(
    clean_48k: np.ndarray,
    noise_48k: np.ndarray | None,
    mixture_48k: np.ndarray,
    sep_32k_raw: np.ndarray,
    native_sr: int = 48000,
    gap_range_samples: list[int] | None = None,
) -> dict[str, Any]:
    """Finalize the exact audition waveform FIRST, then compute all metrics on that exact waveform."""
    raw_48k = align_length(
        librosa.resample(sep_32k_raw.astype(np.float64), orig_sr=32000, target_sr=native_sr, res_type="soxr_hq"),
        len(clean_48k),
    ).astype(np.float32)

    # Apply finalization ONCE before metric evaluation so metrics describe the EXACT auditioned audio!
    audition_48k, clip_info = finalize_audition_waveform(raw_48k)
    residual_48k = (mixture_48k.astype(np.float64) - audition_48k.astype(np.float64)).astype(np.float32)

    # Derive 32 kHz versions from the exact same finalized audition waveform & references
    clean_32k = librosa.resample(clean_48k.astype(np.float64), orig_sr=native_sr, target_sr=32000, res_type="soxr_hq")
    mix_32k = librosa.resample(mixture_48k.astype(np.float64), orig_sr=native_sr, target_sr=32000, res_type="soxr_hq")
    audition_32k = align_length(
        librosa.resample(audition_48k.astype(np.float64), orig_sr=native_sr, target_sr=32000, res_type="soxr_hq"),
        len(clean_32k),
    )

    is_clean = noise_48k is None
    in_snr_32 = compute_snr_db(clean_32k, mix_32k)
    out_snr_32 = compute_snr_db(clean_32k, audition_32k)
    in_sisdr_32 = compute_si_sdr_db(clean_32k, mix_32k)
    out_sisdr_32 = compute_si_sdr_db(clean_32k, audition_32k)

    in_snr_48 = compute_snr_db(clean_48k, mixture_48k)
    out_snr_48 = compute_snr_db(clean_48k, audition_48k)
    in_sisdr_48 = compute_si_sdr_db(clean_48k, mixture_48k)
    out_sisdr_48 = compute_si_sdr_db(clean_48k, audition_48k)

    mr_lsd_32 = compute_multi_res_lsd_db(clean_32k, audition_32k, 32000)
    mr_lsd_48 = compute_multi_res_lsd_db(clean_48k, audition_48k, native_sr)
    onset_corr = compute_onset_preservation(clean_32k, audition_32k, 32000)
    tgt_harm = compute_active_target_and_harmonic_metrics(clean_32k, audition_32k, 32000)
    bg_supp = compute_background_suppression(clean_48k, noise_48k, mixture_48k, audition_48k, gap_range_samples)

    clean_crest = compute_crest_factor_db(clean_32k)
    aud_crest = compute_crest_factor_db(audition_32k)
    crest_delta = round(aud_crest - clean_crest, 2) if (clean_crest is not None and aud_crest is not None) else None

    return {
        "in_snr_32k_db": in_snr_32,
        "out_snr_32k_db": out_snr_32,
        "delta_snr_32k_db": None if (is_clean or in_snr_32 is None or out_snr_32 is None) else round(out_snr_32 - in_snr_32, 2),
        "in_sisdr_32k_db": in_sisdr_32,
        "out_sisdr_32k_db": out_sisdr_32,
        "delta_sisdr_32k_db": None if (is_clean or in_sisdr_32 is None or out_sisdr_32 is None) else round(out_sisdr_32 - in_sisdr_32, 2),
        "out_snr_48k_db": out_snr_48,
        "delta_snr_48k_db": None if (is_clean or in_snr_48 is None or out_snr_48 is None) else round(out_snr_48 - in_snr_48, 2),
        "out_sisdr_48k_db": out_sisdr_48,
        "delta_sisdr_48k_db": None if (is_clean or in_sisdr_48 is None or out_sisdr_48 is None) else round(out_sisdr_48 - in_sisdr_48, 2),
        "mr_lsd_32k_db": mr_lsd_32,
        "mr_lsd_48k_db": mr_lsd_48,
        "onset_preservation": onset_corr,
        "crest_delta_db": crest_delta,
        **tgt_harm,
        **bg_supp,
        **clip_info,
        "raw_48k": raw_48k,
        "audition_48k": audition_48k,
        "residual_48k": residual_48k,
    }


print("[OK] PeakMemoryMonitor, waveform finalizer, and full metric suite loaded.")

## Section 7 — Sequential Evaluation: CLAPSep (7A) & AudioSep with Long-Recording Validation (7B)

Models run sequentially and are completely unloaded between phases to keep VRAM and RAM well within Colab limits.

### Important Upstream Discovery in `Audio-AGI/AudioSep` (`944583f18b`, `models/resunet.py:chunk_inference`)
Direct source inspection of upstream `ResUNet30.chunk_inference` revealed three defects:
1. Returns **all zeros** for any input `<= 5.0 s` (`L <= WINDOW`).
2. Runs inference **twice per loop iteration** because the trailing `if current_idx < L:` block is indented inside the `while` loop, risking boundary artifacts or shape errors on short tails.
3. Returns `float64` ndarray `[1, L]` instead of `float32` `[1, 1, L]`.

Therefore, Cell 7B implements a clean, mathematically verified **overlap-discard chunked wrapper** (`run_audiosep_validated_chunked`) calling `model.ss_model(input_dict)["waveform"]` directly on sliding 5.0 s windows (`NL=1.0 s`, `NC=3.0 s`, `NR=1.0 s`), AND executes an explicit **short (`3.5 s`) + long (`13.5 s`) validation self-test** verifying `dtype`, `ndim`, exact sample count, sample alignment, and absence of missing/duplicated chunk-boundary samples before running the benchmark.

In [ ]:
# === CELL 7A: RECOVER ARCHIVED CLAPSEP RESULTS (NO MODEL RE-RUN) ===# The 70 CLAPSep WAV triplets were already generated and persisted in outputs/clapsep.# This cell reconstructs objective metrics from archived raw predictions. Because the# original 32 kHz arrays are unavailable after a runtime reset, raw 48 kHz WAVs are# resampled back to 32 kHz; the resulting metrics are labelled RECONSTRUCTED.import jsonimport refrom pathlib import Pathimport librosaimport numpy as npimport soundfile as sfOUTPUTS_DIR = WORK_ROOT / "outputs"CLAPSEP_OUT_DIR = OUTPUTS_DIR / "clapsep"AUDIOSEP_OUT_DIR = OUTPUTS_DIR / "audiosep"CLAPSEP_OUT_DIR.mkdir(parents=True, exist_ok=True)AUDIOSEP_OUT_DIR.mkdir(parents=True, exist_ok=True)CLAPSEP_QUERIES = [    {"query_id": "clapsep_ukulele", "pos": "ukulele", "neg": ""},    {"query_id": "clapsep_ukulele_strumming", "pos": "ukulele strumming", "neg": ""},    {"query_id": "clapsep_neg_speech", "pos": "ukulele strumming", "neg": "human speech, talking"},    {"query_id": "clapsep_neg_env", "pos": "ukulele strumming", "neg": "traffic, wind, fan noise, background noise"},    {"query_id": "clapsep_neg_guitar", "pos": "ukulele", "neg": "acoustic guitar"},]CLAPSEP_REAL_RESULTS = []CLAPSEP_SYNTH_RESULTS = []CLAPSEP_LOAD_STATS = {    "engine": "CLAPSep",    "execution_status": "recovered_from_archived_outputs",    "model_load_and_inference_telemetry": "not_recoverable_after_runtime_reset",    "metric_provenance": "RECONSTRUCTED from archived 48 kHz raw predictions by resampling to 32 kHz",}if not CLAPSEP_OUT_DIR.exists():    raise FileNotFoundError(f"Archived CLAPSep output directory is missing: {CLAPSEP_OUT_DIR}")# Restore the deterministic scenario metadata from the saved input WAVs.SYNTH_DIR = WORK_ROOT / "inputs" / "synthetic_benchmark"clean_path = SYNTH_DIR / "clean_ukulele_48k.wav"if not clean_path.is_file():    raise FileNotFoundError(f"Saved synthetic clean reference is missing: {clean_path}")SYNTHETIC_SCENARIOS = [{    "scenario_id": "clean_ukulele",    "noise_type": "none",    "interference_source": "none",    "label": "Clean Ukulele (regression guard)",    "target_snr_db": None,    "clean_path": str(clean_path),    "noise_path": None,    "mixture_path": str(clean_path),    "sample_rate": 48000,    "gap_range_samples": [199200, 247200],}]_noise_labels = {    "speech_proxy": "Synthetic Speech Proxy (formant DSP)",    "traffic": "Traffic",    "fan": "Fan Noise",    "wind": "Wind",    "pink_noise": "Pink Noise",    "acoustic_guitar": "Acoustic Guitar",}for _noise in _noise_labels:    for _snr in (0, 6):        _sid = f"ukulele_plus_{_noise}_snr{_snr}db"        _clean = SYNTH_DIR / f"{_sid}__clean.wav"        _noise_path = SYNTH_DIR / f"{_sid}__noise.wav"        _mix = SYNTH_DIR / f"{_sid}__mixture.wav"        if _clean.is_file() and _noise_path.is_file() and _mix.is_file():            SYNTHETIC_SCENARIOS.append({                "scenario_id": _sid,                "noise_type": _noise,                "interference_source": "synthetic_dsp_proxy",                "label": f"Ukulele + {_noise_labels[_noise]} ({_snr:+d} dB SNR)",                "target_snr_db": float(_snr),                "clean_path": str(_clean),                "noise_path": str(_noise_path),                "mixture_path": str(_mix),                "sample_rate": 48000,                "gap_range_samples": [199200, 247200],            })def _clapsep_artifact_paths(prefix: str, query_id: str) -> dict:    base = CLAPSEP_OUT_DIR / f"{prefix}__{query_id}"    return {        "raw_f32_wav": str(base.with_name(base.name + "__raw_f32.wav")),        "audition_wav": str(base.with_name(base.name + "__audition.wav")),        "residual_diff_wav": str(base.with_name(base.name + "__diff_residual.wav")),    }def _reconstruct_clapsep_record(sc: dict, q: dict) -> dict:    paths = _clapsep_artifact_paths(sc["scenario_id"], q["query_id"])    for _key, _path in paths.items():        if not Path(_path).is_file():            raise FileNotFoundError(f"Missing archived CLAPSep artifact ({_key}): {_path}")    clean_48k, sr = sf.read(sc["clean_path"], dtype="float32")    mixture_48k, _ = sf.read(sc["mixture_path"], dtype="float32")    noise_48k = sf.read(sc["noise_path"], dtype="float32")[0] if sc["noise_path"] else None    raw_48k, raw_sr = sf.read(paths["raw_f32_wav"], dtype="float32")    if raw_sr != sr:        raise ValueError(f"Sample-rate mismatch for {paths['raw_f32_wav']}: {raw_sr} != {sr}")    raw_32k = librosa.resample(        np.asarray(raw_48k, dtype=np.float64), orig_sr=sr, target_sr=32000, res_type="soxr_hq"    ).astype(np.float32)    ev = evaluate_calibrated_result(        clean_48k=clean_48k,        noise_48k=noise_48k,        mixture_48k=mixture_48k,        sep_32k_raw=raw_32k,        native_sr=sr,        gap_range_samples=sc["gap_range_samples"],    )    # Keep the original archived paths; do not overwrite the original inference artifacts.    ev.pop("raw_48k", None)    ev.pop("audition_48k", None)    ev.pop("residual_48k", None)    return {        "engine": "CLAPSep",        "scenario_id": sc["scenario_id"],        "noise_type": sc["noise_type"],        "interference_source": sc["interference_source"],        "target_snr_db": sc["target_snr_db"],        "query_id": q["query_id"],        "positive_query": q["pos"],        "negative_query": q["neg"] or "(none)",        **paths,        **ev,        "inference_time_s": None,        "rtf": None,        "peak_ram_during_inference_mb": None,        "gpu_peak_allocated_mb": None,        "gpu_peak_reserved_mb": None,        "metric_status": "RECONSTRUCTED_FROM_ARCHIVED_48KHZ_RAW",    }_expected = len(SYNTHETIC_SCENARIOS) * len(CLAPSEP_QUERIES)for _sc in SYNTHETIC_SCENARIOS:    for _q in CLAPSEP_QUERIES:        CLAPSEP_SYNTH_RESULTS.append(_reconstruct_clapsep_record(_sc, _q))# Real recording: retain the archived output and report level/clipping telemetry only.REAL_INPUT_DIR = WORK_ROOT / "inputs" / "real_ukulele"_real_manifest = REAL_INPUT_DIR / "real_recording_manifest.json"REAL_RECORDING_INFO = json.loads(_real_manifest.read_text(encoding="utf-8")) if _real_manifest.is_file() else Noneif REAL_RECORDING_INFO is not None:    _real_native, _real_sr = sf.read(REAL_RECORDING_INFO["decoded_native_wav_path"], dtype="float32", always_2d=True)    _real_mono = np.mean(_real_native, axis=1, dtype=np.float64)    for _q in CLAPSEP_QUERIES:        _paths = _clapsep_artifact_paths("real", _q["query_id"])        if not all(Path(p).is_file() for p in _paths.values()):            continue        _raw, _sr = sf.read(_paths["raw_f32_wav"], dtype="float32")        _audition, _ = sf.read(_paths["audition_wav"], dtype="float32")        _residual, _ = sf.read(_paths["residual_diff_wav"], dtype="float32")        _peak = float(np.max(np.abs(_raw))) if _raw.size else 0.0        _clipped = int(np.sum(np.abs(_raw) > 1.0))        _rms = lambda x: float(20.0 * np.log10(max(np.sqrt(np.mean(np.asarray(x, dtype=np.float64) ** 2)), 1e-12)))        CLAPSEP_REAL_RESULTS.append({            "engine": "CLAPSep",            "query_id": _q["query_id"],            "positive_query": _q["pos"],            "negative_query": _q["neg"] or "(none)",            "native_sr": int(_sr),            "native_channels": 1,            "internal_sr": 32000,            "internal_channels": 1,            "input_rms_dbfs": round(_rms(_real_mono), 2),            "output_rms_dbfs": round(_rms(_audition), 2),            "residual_diff_rms_dbfs": round(_rms(_residual), 2),            "raw_peak_linear": round(_peak, 6),            "was_clipped": _clipped > 0,            "clipped_samples": _clipped,            "clipped_fraction": round(_clipped / max(len(_raw), 1), 6),            "gain_adjustment_db": 0.0,            **_paths,            "inference_time_s": None,            "rtf": None,            "peak_ram_during_inference_mb": None,            "gpu_peak_allocated_mb": None,            "gpu_peak_reserved_mb": None,            "metric_status": "ARCHIVED_OUTPUT_LEVEL_TELEMETRY_ONLY_NO_GROUND_TRUTH",        })print(f"[OK] Recovered {len(CLAPSEP_SYNTH_RESULTS)} synthetic CLAPSep metric rows "      f"({len(SYNTHETIC_SCENARIOS)} scenarios x {len(CLAPSEP_QUERIES)} prompts).")print(f"[OK] Recovered {len(CLAPSEP_REAL_RESULTS)} real-recording CLAPSep output records.")print("NOTE: CLAPSep metrics are reconstructed from archived 48 kHz raw WAVs; original inference-time and peak-memory telemetry cannot be restored.")if len(CLAPSEP_SYNTH_RESULTS) != _expected:    raise RuntimeError(f"Expected {_expected} synthetic CLAPSep rows, recovered {len(CLAPSEP_SYNTH_RESULTS)}.")

In [ ]:
# === CELL 7B: RUN AUDIOSEP (SCOPED SAFE LOAD + SHORT/LONG CHUNK VALIDATION + UNLOAD) ===
import gc
import os
import sys
import time
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import torch

AUDIOSEP_OUT_DIR = OUTPUTS_DIR / "audiosep"
AUDIOSEP_OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIOSEP_QUERIES = [
    {"query_id": "audiosep_ukulele", "pos": "ukulele", "neg": "N/A (unsupported)"},
    {"query_id": "audiosep_ukulele_strumming", "pos": "ukulele strumming", "neg": "N/A (unsupported)"},
]

AUDIOSEP_REAL_RESULTS = []
AUDIOSEP_SYNTH_RESULTS = []
AUDIOSEP_LOAD_STATS = {}
AUDIOSEP_CHUNK_VALIDATION = {}

if not DOWNLOAD_AUDIOSEP:
    print("DOWNLOAD_AUDIOSEP is False — skipping AudioSep evaluation.")
else:
    prev_cwd = Path.cwd()
    os.chdir(AUDIOSEP_REPO)
    for stale_mod in ("pipeline", "utils", "models", "models.clap_encoder"):
        sys.modules.pop(stale_mod, None)
    if str(AUDIOSEP_REPO) not in sys.path:
        sys.path.insert(0, str(AUDIOSEP_REPO))

    import models.CLAP.open_clip.factory as audiosep_clap_factory

    _orig_as_load_sd = audiosep_clap_factory.load_state_dict

    def _safe_as_load_state_dict(checkpoint_path: str, map_location="cpu", skip_params=True):
        sd = _orig_as_load_sd(checkpoint_path, map_location=map_location, skip_params=skip_params)
        for bad_key in [k for k in list(sd.keys()) if k.endswith("embeddings.position_ids")]:
            sd.pop(bad_key, None)
        return sd

    audiosep_clap_factory.load_state_dict = _safe_as_load_state_dict

    from pipeline import build_audiosep

    as_ckpt = AUDIOSEP_CKPT_DIR / "audiosep_base_4M_steps.ckpt"
    as_clap_ckpt = AUDIOSEP_CKPT_DIR / "music_speech_audioset_epoch_15_esc_89.98.pt"

    t_load0 = time.perf_counter()
    with PeakMemoryMonitor() as load_mem:
        with scoped_trusted_torch_load({as_ckpt, as_clap_ckpt}, CHECKPOINT_LOAD_AUDIT_LOG):
            audiosep_model = build_audiosep(
                config_yaml=str(AUDIOSEP_REPO / "config" / "audiosep_base.yaml"),
                checkpoint_path=str(as_ckpt),
                device=torch.device(DEVICE),
            )
    load_s = round(time.perf_counter() - t_load0, 2)

    AUDIOSEP_LOAD_STATS = {
        "engine": "AudioSep",
        "device": DEVICE,
        "load_time_s": load_s,
        "ram_before_load_mb": load_mem.ram_before_mb,
        "ram_after_load_mb": load_mem.ram_after_mb,
        "peak_ram_during_load_mb": load_mem.peak_ram_mb,
        "process_vmhwm_mb": process_vmhwm_mb(),
        "gpu_peak_allocated_load_mb": load_mem.gpu_peak_allocated_mb,
        "gpu_peak_reserved_load_mb": load_mem.gpu_peak_reserved_mb,
        "internal_sr": 32000,
        "internal_channels": 1,
    }
    print(
        f"[OK] Loaded AudioSep on {DEVICE} in {load_s:.2f}s | "
        f"RAM before={load_mem.ram_before_mb} MB, after={load_mem.ram_after_mb} MB, peak_load={load_mem.peak_ram_mb} MB | "
        f"VRAM alloc={load_mem.gpu_peak_allocated_mb} MB, reserved={load_mem.gpu_peak_reserved_mb} MB"
    )


    def _run_audiosep_direct(wav_32k: np.ndarray, conditions: torch.Tensor) -> np.ndarray:
        """Direct single-pass ResUNet30 inference on 32 kHz mono float32 audio."""
        inp = {
            "mixture": torch.from_numpy(wav_32k.astype(np.float32))[None, None, :].to(DEVICE),
            "condition": conditions.to(DEVICE),
        }
        with torch.no_grad():
            out_t = audiosep_model.ss_model(inp)["waveform"]
        return out_t.squeeze(0).squeeze(0).detach().cpu().numpy().astype(np.float32)


    def _run_audiosep_validated_chunked(
        wav_32k: np.ndarray,
        conditions: torch.Tensor,
        nl: int = 32000,
        nc: int = 96000,
        nr: int = 32000,
    ) -> np.ndarray:
        """Validated overlap-discard chunking (1s left + 3s center + 1s right = 5s window).

        Replaces upstream models/resunet.py:chunk_inference (which returns zeros for <=5s,
        double-runs inside its while loop, and returns float64 [1, L]).
        """
        x = np.asarray(wav_32k, dtype=np.float32).reshape(-1)
        orig_len = len(x)
        if orig_len == 0:
            return np.zeros(0, dtype=np.float32)

        win = nl + nc + nr  # 160,000 samples (5.0 s)
        # Pad left by nl and right by enough to cover the last full center hop + nr
        n_hops = int(np.ceil(orig_len / nc))
        padded_right = (n_hops * nc - orig_len) + nr
        x_pad = np.pad(x, (nl, padded_right), mode="reflect" if orig_len > max(nl, padded_right) else "constant")

        out = np.zeros(n_hops * nc, dtype=np.float32)
        for h in range(n_hops):
            start = h * nc
            seg = x_pad[start : start + win]
            assert len(seg) == win, f"Unexpected chunk length: {len(seg)} != {win}"
            seg_out = _run_audiosep_direct(seg, conditions)
            assert seg_out.ndim == 1 and len(seg_out) == win, (
                f"Chunk output shape mismatch: {seg_out.shape} vs ({win},)"
            )
            out[h * nc : (h + 1) * nc] = seg_out[nl : nl + nc]

        return out[:orig_len].astype(np.float32)


    def validate_audiosep_short_and_long() -> dict:
        """Validate dtype, shape, exact sample count, sample alignment, and chunk boundaries for short & long inputs."""
        with torch.no_grad():
            cond = audiosep_model.query_encoder.get_query_embed(modality="text", text=["ukulele"], device=DEVICE)

        # 1) Short input (3.5 s = 112,000 samples, < 5s & < 10s)
        short_len = 112000
        t_s = np.arange(short_len, dtype=np.float32) / 32000.0
        short_sig = (0.35 * np.sin(2.0 * np.pi * 440.0 * t_s) * np.exp(-0.4 * t_s)).astype(np.float32)
        short_direct = _run_audiosep_direct(short_sig, cond)
        short_chunked = _run_audiosep_validated_chunked(short_sig, cond)

        assert short_direct.dtype == np.float32 and short_direct.ndim == 1 and len(short_direct) == short_len
        assert short_chunked.dtype == np.float32 and short_chunked.ndim == 1 and len(short_chunked) == short_len

        def _assert_finite_non_silent(label: str, input_wave: np.ndarray, output_wave: np.ndarray) -> dict:
            """Fail on NaN/Inf or effectively silent output; retain a permissive -80 dB relative floor."""
            if not np.isfinite(output_wave).all():
                raise RuntimeError(f"{label}: output contains NaN or infinite samples.")
            input_rms = float(np.sqrt(np.mean(np.square(input_wave, dtype=np.float64))))
            output_rms = float(np.sqrt(np.mean(np.square(output_wave, dtype=np.float64))))
            output_peak = float(np.max(np.abs(output_wave))) if output_wave.size else 0.0
            min_rms = max(1e-8, input_rms * 1e-4)
            if output_rms < min_rms or output_peak < 1e-7:
                raise RuntimeError(
                    f"{label}: output is silent/nearly silent "
                    f"(input RMS={input_rms:.3e}, output RMS={output_rms:.3e}, "
                    f"peak={output_peak:.3e}, required RMS>={min_rms:.3e})."
                )
            return {
                "input_rms": input_rms,
                "output_rms": output_rms,
                "output_peak": output_peak,
                "output_to_input_rms_ratio": output_rms / max(input_rms, 1e-12),
            }

        short_direct_energy = _assert_finite_non_silent("Short direct path", short_sig, short_direct)
        short_chunked_energy = _assert_finite_non_silent("Short chunked path", short_sig, short_chunked)
        short_corr = float(np.corrcoef(short_direct, short_chunked)[0, 1]) if (np.std(short_direct) > 1e-7 and np.std(short_chunked) > 1e-7) else None

        # 2) Long input (13.5 s = 432,000 samples, > 10 s, spans 5 center hops at 3s, 6s, 9s, 12s)
        long_len = 432000
        t_l = np.arange(long_len, dtype=np.float32) / 32000.0
        # Place distinct ukulele-band tonal bursts straddling exact chunk boundaries (3.0s, 6.0s, 9.0s, 12.0s)
        long_sig = np.zeros(long_len, dtype=np.float32)
        for center_s in [1.5, 3.0, 6.0, 9.0, 12.0]:
            env = np.exp(-0.5 * ((t_l - center_s) / 0.15) ** 2)
            long_sig += (0.3 * env * np.sin(2.0 * np.pi * 392.0 * t_l)).astype(np.float32)

        long_direct = _run_audiosep_direct(long_sig, cond)
        long_chunked = _run_audiosep_validated_chunked(long_sig, cond)

        assert long_direct.dtype == np.float32 and long_direct.ndim == 1 and len(long_direct) == long_len
        assert long_chunked.dtype == np.float32 and long_chunked.ndim == 1 and len(long_chunked) == long_len
        long_direct_energy = _assert_finite_non_silent("Long direct path", long_sig, long_direct)
        long_chunked_energy = _assert_finite_non_silent("Long chunked path", long_sig, long_chunked)

        # Check sample alignment lag via cross-correlation around zero lag (+/- 32 samples)
        best_lag = 0
        best_dot = -1e30
        if float(np.std(long_direct)) > 1e-7 and float(np.std(long_chunked)) > 1e-7:
            for lag in range(-32, 33):
                a = long_direct[32:-32]
                b = long_chunked[32 + lag : long_len - 32 + lag]
                d = float(np.dot(a, b))
                if d > best_dot:
                    best_dot = d
                    best_lag = lag
            long_corr = float(np.corrcoef(long_direct, long_chunked)[0, 1])
        else:
            long_corr = None

        # Check chunk-boundary sample jumps at exact 3s, 6s, 9s, 12s seam points vs local median step
        seam_indices = [96000, 192000, 288000, 384000]
        seam_steps = [float(abs(long_chunked[idx] - long_chunked[idx - 1])) for idx in seam_indices]
        typical_step = float(np.percentile(np.abs(np.diff(long_chunked)), 99)) + 1e-9
        max_seam_ratio = round(max(seam_steps) / typical_step, 3)

        report = {
            "short_3p5s_dtype": str(short_chunked.dtype),
            "short_3p5s_shape": list(short_chunked.shape),
            "short_direct_vs_chunked_corr": round(short_corr, 4) if short_corr is not None else None,
            "short_direct_energy": short_direct_energy,
            "short_chunked_energy": short_chunked_energy,
            "long_13p5s_dtype": str(long_chunked.dtype),
            "long_13p5s_shape": list(long_chunked.shape),
            "long_direct_vs_chunked_corr": round(long_corr, 4) if long_corr is not None else None,
            "long_direct_energy": long_direct_energy,
            "long_chunked_energy": long_chunked_energy,
            "long_alignment_lag_samples": best_lag,
            "long_max_seam_step_ratio": max_seam_ratio,
            "upstream_chunk_inference_note": (
                "Bypassed upstream resunet.py:chunk_inference due to verified defects "
                "(all-zero output for <=5s, duplicate inference in while loop, float64 [1,L] shape)."
            ),
        }
        if best_lag != 0:
            raise RuntimeError(f"AudioSep chunked alignment lag non-zero: {best_lag} samples!")
        return report


    AUDIOSEP_CHUNK_VALIDATION = validate_audiosep_short_and_long()
    print("[OK] AudioSep short (3.5s) & long (13.5s) chunk validation PASSED:")
    for k, v in AUDIOSEP_CHUNK_VALIDATION.items():
        print(f"  {k:32s}: {v}")


    def run_audiosep_waveform_32k(wav_32k: np.ndarray, text_query: str) -> tuple[np.ndarray, dict]:
        orig_len = len(wav_32k)
        t0 = time.perf_counter()
        with PeakMemoryMonitor() as inf_mem:
            with torch.no_grad():
                conditions = audiosep_model.query_encoder.get_query_embed(
                    modality="text",
                    text=[text_query],
                    device=DEVICE,
                )
            if orig_len > 320000:
                sep_out = _run_audiosep_validated_chunked(wav_32k, conditions)
            else:
                sep_out = _run_audiosep_direct(wav_32k, conditions)
        infer_s = time.perf_counter() - t0

        assert sep_out.dtype == np.float32 and sep_out.ndim == 1 and len(sep_out) == orig_len, (
            f"AudioSep output contract violation: dtype={sep_out.dtype}, shape={sep_out.shape}, expected=({orig_len},)"
        )
        audio_dur_s = orig_len / 32000.0
        stats = {
            "inference_time_s": round(infer_s, 3),
            "rtf": round(infer_s / max(audio_dur_s, 1e-6), 4),
            "peak_ram_during_inference_mb": inf_mem.peak_ram_mb,
            "gpu_peak_allocated_mb": inf_mem.gpu_peak_allocated_mb,
            "gpu_peak_reserved_mb": inf_mem.gpu_peak_reserved_mb,
            "used_chunked_path": orig_len > 320000,
        }
        return sep_out, stats


    # --- 1) Real Ukulele Recording (if uploaded) ---
    if REAL_RECORDING_INFO is not None:
        real_native, real_sr = sf.read(REAL_RECORDING_INFO["decoded_native_wav_path"], dtype="float32", always_2d=True)
        real_native_mono = np.mean(real_native, axis=1, dtype=np.float64)
        real_32k, _ = sf.read(REAL_RECORDING_INFO["input_32k_mono_wav_path"], dtype="float32")
        print(f"\nRunning AudioSep on REAL recording ({REAL_RECORDING_INFO['duration_seconds']}s, {real_sr} Hz {real_native.shape[1]}ch -> 32 kHz mono)...")
        for q in AUDIOSEP_QUERIES:
            sep_32k, st = run_audiosep_waveform_32k(real_32k, q["pos"])
            raw_native = align_length(
                librosa.resample(sep_32k.astype(np.float64), orig_sr=32000, target_sr=real_sr, res_type="soxr_hq"),
                len(real_native_mono),
            ).astype(np.float32)
            audition_native, clip_info = finalize_audition_waveform(raw_native)
            residual_native = (real_native_mono - audition_native.astype(np.float64)).astype(np.float32)

            paths = save_audio_artifacts(
                raw_native, audition_native, residual_native, real_sr, AUDIOSEP_OUT_DIR / f"real__{q['query_id']}"
            )
            in_rms_db = round(float(20 * np.log10(max(np.sqrt(np.mean(real_native_mono ** 2)), 1e-12))), 2)
            out_rms_db = round(float(20 * np.log10(max(np.sqrt(np.mean(audition_native.astype(np.float64) ** 2)), 1e-12))), 2)
            res_rms_db = round(float(20 * np.log10(max(np.sqrt(np.mean(residual_native.astype(np.float64) ** 2)), 1e-12))), 2)

            AUDIOSEP_REAL_RESULTS.append(
                {
                    "engine": "AudioSep",
                    "query_id": q["query_id"],
                    "positive_query": q["pos"],
                    "negative_query": q["neg"],
                    "native_sr": real_sr,
                    "native_channels": int(real_native.shape[1]),
                    "internal_sr": 32000,
                    "internal_channels": 1,
                    "input_rms_dbfs": in_rms_db,
                    "output_rms_dbfs": out_rms_db,
                    "residual_diff_rms_dbfs": res_rms_db,
                    **clip_info,
                    **paths,
                    **st,
                }
            )
            print(f"  {q['query_id']:26s} | {st['inference_time_s']:5.2f}s (RTF={st['rtf']:.3f}) | clipped={clip_info['was_clipped']} | peak_inf_RAM={st['peak_ram_during_inference_mb']} MB")

    # --- 2) Calibrated Synthetic + Optional Real-Speech Scenarios ---
    print(f"\nRunning AudioSep across {len(SYNTHETIC_SCENARIOS)} calibrated scenarios x {len(AUDIOSEP_QUERIES)} queries...")
    for sc in SYNTHETIC_SCENARIOS:
        clean_48k, sr = sf.read(sc["clean_path"], dtype="float32")
        mix_48k, _ = sf.read(sc["mixture_path"], dtype="float32")
        noise_48k = sf.read(sc["noise_path"], dtype="float32")[0] if sc["noise_path"] else None
        mix_32k = librosa.resample(mix_48k.astype(np.float64), orig_sr=sr, target_sr=32000, res_type="soxr_hq").astype(np.float32)

        for q in AUDIOSEP_QUERIES:
            sep_32k, st = run_audiosep_waveform_32k(mix_32k, q["pos"])
            ev = evaluate_calibrated_result(
                clean_48k=clean_48k,
                noise_48k=noise_48k,
                mixture_48k=mix_48k,
                sep_32k_raw=sep_32k,
                native_sr=sr,
                gap_range_samples=sc["gap_range_samples"],
            )
            paths = save_audio_artifacts(
                ev.pop("raw_48k"), ev.pop("audition_48k"), ev.pop("residual_48k"), sr,
                AUDIOSEP_OUT_DIR / f"{sc['scenario_id']}__{q['query_id']}",
            )
            AUDIOSEP_SYNTH_RESULTS.append(
                {
                    "engine": "AudioSep",
                    "scenario_id": sc["scenario_id"],
                    "noise_type": sc["noise_type"],
                    "interference_source": sc["interference_source"],
                    "target_snr_db": sc["target_snr_db"],
                    "query_id": q["query_id"],
                    "positive_query": q["pos"],
                    "negative_query": q["neg"],
                    **paths,
                    **ev,
                    **st,
                }
            )
        print(f"  Completed scenario: {sc['scenario_id']}")

    os.chdir(prev_cwd)
    del audiosep_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"\n[OK] AudioSep complete & unloaded. Current RSS={current_rss_mb()} MB | VmHWM={process_vmhwm_mb()} MB")

## Section 8 — Listening Comparison Deck & Spectrograms

Every `Audio(...)` player below plays the **exact `*_audition.wav` waveform** that produced the objective metrics in Section 9 (zero mismatch between evaluated and auditioned audio).

> **Residual Disclosure:** Every `*_diff_residual.wav` is computed strictly as `input_aligned[n] - audition_output[n]` with **no independent normalization**. It contains all differences introduced by separation — **including any lost ukulele strums, harmonics, or >16 kHz air** — and is **NOT** guaranteed to be pure background noise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, HTML, display
from scipy import signal


def plot_spectrogram_comparison(items: list[tuple[str, str]], title: str):
    n = len(items)
    fig, axes = plt.subplots(n, 1, figsize=(12, 2.7 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, (label, path) in zip(axes, items):
        y, sr = sf.read(path, dtype="float32")
        if y.ndim > 1:
            y = np.mean(y, axis=1)
        f, t, z = signal.stft(y, fs=sr, nperseg=1024, noverlap=768)
        db = 20.0 * np.log10(np.abs(z) + 1e-7)
        ax.pcolormesh(t, f / 1000.0, db, vmin=-85, vmax=-10, shading="auto", cmap="magma")
        ax.axhline(16.0, color="cyan", linestyle="--", linewidth=0.9, alpha=0.7)
        ax.set_ylabel("Freq (kHz)")
        ax.set_ylim(0, min(24.0, sr / 2000.0))
        ax.set_title(label, fontsize=10, loc="left")
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    plt.show()


def play_wav_exact(path: str):
    """Play float32 WAV in Colab without automatic gain normalization so true relative levels are preserved."""
    y, sr = sf.read(path, dtype="float32")
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    if peak > 1.0:
        print(f"  [Playback note] {Path(path).name} peaked at {peak:.3f}; clipped to [-1.0, 1.0] for browser playback only.")
        y = np.clip(y, -1.0, 1.0)
    if y.ndim == 2:
        y = y.T  # IPython.display.Audio expects (channels, samples) for stereo arrays
    display(Audio(data=y, rate=sr, normalize=False))


ALL_REAL_RESULTS = CLAPSEP_REAL_RESULTS + AUDIOSEP_REAL_RESULTS
ALL_SYNTH_RESULTS = CLAPSEP_SYNTH_RESULTS + AUDIOSEP_SYNTH_RESULTS

if REAL_RECORDING_INFO is not None and ALL_REAL_RESULTS:
    display(HTML("<h3>Real Ukulele Recording — Direct Listening Comparison</h3>"))
    display(HTML("<p><b>Original (Native Decoded)</b> — untouched original audio:</p>"))
    play_wav_exact(REAL_RECORDING_INFO["decoded_native_wav_path"])

    spec_items = [("Original (Native SR)", REAL_RECORDING_INFO["decoded_native_wav_path"])]
    for rec in ALL_REAL_RESULTS:
        hdr = (
            f"<b>{rec['engine']}</b> | <code>{rec['query_id']}</code> | "
            f"pos=<code>{rec['positive_query']}</code> | neg=<code>{rec['negative_query']}</code> | "
            f"clipped={rec['was_clipped']}"
        )
        display(HTML(f"<p>{hdr}</p>"))
        play_wav_exact(rec["audition_wav"])
        display(HTML("<small>Processing Difference / Residual (<code>original_mono - audition</code>; NOT guaranteed pure noise):</small>"))
        play_wav_exact(rec["residual_diff_wav"])
        if rec["query_id"] in ("clapsep_ukulele_strumming", "clapsep_neg_env", "audiosep_ukulele_strumming"):
            spec_items.append((f"{rec['engine']} ({rec['query_id']})", rec["audition_wav"]))

    plot_spectrogram_comparison(spec_items, "Real Ukulele Recording — Spectrogram Comparison (Cyan Dashed Line = 16 kHz Model Cap)")
else:
    print("No real ukulele recording was uploaded in Section 4 — showing calibrated 0 dB listening comparisons below.")

PREVIEW_SCENARIOS = [
    "ukulele_plus_real_speech_snr0db",
    "ukulele_plus_speech_proxy_snr0db",
    "ukulele_plus_traffic_snr0db",
    "ukulele_plus_acoustic_guitar_snr0db",
]
PREVIEW_QUERIES = {
    "clapsep_ukulele_strumming",
    "clapsep_neg_speech",
    "clapsep_neg_env",
    "clapsep_neg_guitar",
    "audiosep_ukulele_strumming",
}

for sid in PREVIEW_SCENARIOS:
    sc_match = [s for s in SYNTHETIC_SCENARIOS if s["scenario_id"] == sid]
    if not sc_match:
        continue
    sc = sc_match[0]
    display(HTML(f"<h3>Calibrated Listening Check [{sc['interference_source']}]: {sc['label']}</h3>"))
    display(HTML("<small>Clean Ukulele Reference | Noisy Mixture (0 dB SNR):</small>"))
    play_wav_exact(sc["clean_path"])
    play_wav_exact(sc["mixture_path"])
    for rec in [r for r in ALL_SYNTH_RESULTS if r["scenario_id"] == sid and r["query_id"] in PREVIEW_QUERIES]:
        display(HTML(
            f"<small><b>{rec['engine']}</b> — <code>{rec['query_id']}</code> "
            f"(ΔSI-SDR 32k={rec['delta_sisdr_32k_db']} dB, MR-LSD={rec['mr_lsd_32k_db']} dB, "
            f"level_ret={rec['target_level_retention_db']} dB, clipped={rec['was_clipped']}):</small>"
        ))
        play_wav_exact(rec["audition_wav"])

## Section 9 — Comparison Tables, Security & Memory Telemetry Report & Downloadable Bundle

Exports all metrics, manifests, checkpoint load audit logs, listening scorecard template, Markdown decision report, and comparison audio files into `sonora_stage5_cloud_benchmark_results.zip`.

In [ ]:
import json
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

REPORT_DIR = WORK_ROOT / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df_synth = pd.DataFrame(ALL_SYNTH_RESULTS)
df_real = pd.DataFrame(ALL_REAL_RESULTS)
df_load = pd.DataFrame([x for x in [CLAPSEP_LOAD_STATS, AUDIOSEP_LOAD_STATS] if x])


def _safe_md_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "_Not executed._"
    try:
        return df.to_markdown(index=False)
    except Exception:
        return "```\n" + df.to_string(index=False) + "\n```"


print("=== TABLE 1: MODEL LOAD & GENUINE PEAK MEMORY TELEMETRY ===")
if not df_load.empty:
    display(df_load)

print("\n=== TABLE 2: CLEAN UKULELE REGRESSION GUARD (TARGET PRESERVATION — NO NOISE) ===")
if not df_synth.empty:
    clean_cols = [
        "engine", "query_id", "out_sisdr_32k_db", "out_sisdr_48k_db",
        "mr_lsd_32k_db", "mr_lsd_48k_db", "active_frame_spectral_corr",
        "target_level_retention_db", "onset_preservation", "harmonic_cosine_sim",
        "crest_delta_db", "was_clipped", "rtf",
        "peak_ram_during_inference_mb", "gpu_peak_allocated_mb", "gpu_peak_reserved_mb",
    ]
    df_clean = df_synth[df_synth["noise_type"] == "none"][clean_cols]
    display(df_clean)
else:
    df_clean = pd.DataFrame()

print("\n=== TABLE 3: CALIBRATED NOISY MIXTURES SUMMARY (SEPARATED BY INTERFERENCE SOURCE) ===")
if not df_synth.empty:
    noisy_df = df_synth[df_synth["noise_type"] != "none"]
    summary_by_query = (
        noisy_df.groupby(
            ["interference_source", "engine", "query_id", "positive_query", "negative_query"], dropna=False
        )[
            [
                "delta_snr_32k_db",
                "delta_sisdr_32k_db",
                "delta_sisdr_48k_db",
                "noise_projection_suppression_db",
                "gap_noise_suppression_db",
                "mr_lsd_32k_db",
                "active_frame_spectral_corr",
                "target_level_retention_db",
                "onset_preservation",
                "harmonic_cosine_sim",
                "rtf",
                "peak_ram_during_inference_mb",
                "gpu_peak_allocated_mb",
                "gpu_peak_reserved_mb",
            ]
        ]
        .mean(numeric_only=True)
        .round(3)
        .reset_index()
    )
    display(summary_by_query)

    print("\n=== TABLE 4: BEST QUERY PER ENGINE BY NOISE TYPE & SNR (32 kHz ΔSI-SDR) ===")
    best_by_noise = (
        noisy_df.sort_values("delta_sisdr_32k_db", ascending=False)
        .groupby(["interference_source", "noise_type", "target_snr_db", "engine"], as_index=False)
        .first()[
            [
                "interference_source", "noise_type", "target_snr_db", "engine", "query_id",
                "in_sisdr_32k_db", "out_sisdr_32k_db", "delta_sisdr_32k_db", "delta_sisdr_48k_db",
                "noise_projection_suppression_db", "gap_noise_suppression_db",
                "mr_lsd_32k_db", "active_frame_spectral_corr", "target_level_retention_db",
                "onset_preservation", "harmonic_cosine_sim", "was_clipped",
            ]
        ]
    )
    display(best_by_noise)
else:
    summary_by_query = pd.DataFrame()
    best_by_noise = pd.DataFrame()

print("\n=== TABLE 5: REAL UKULELE RECORDING TELEMETRY (LISTENING REQUIRED — NO GROUND TRUTH) ===")
if not df_real.empty:
    real_cols = [
        "engine", "query_id", "positive_query", "negative_query",
        "input_rms_dbfs", "output_rms_dbfs", "residual_diff_rms_dbfs",
        "raw_peak_linear", "was_clipped", "clipped_samples",
        "inference_time_s", "rtf",
        "peak_ram_during_inference_mb", "gpu_peak_allocated_mb", "gpu_peak_reserved_mb",
    ]
    display(df_real[real_cols])
else:
    print("Real recording track was not uploaded.")

# Persist CSVs & JSON audit logs
df_synth.to_csv(REPORT_DIR / "metrics_calibrated.csv", index=False)
df_real.to_csv(REPORT_DIR / "metrics_real.csv", index=False)
df_load.to_csv(REPORT_DIR / "model_load_telemetry.csv", index=False)
(REPORT_DIR / "checkpoint_load_audit.json").write_text(
    json.dumps(CHECKPOINT_LOAD_AUDIT_LOG, indent=2), encoding="utf-8"
)
(REPORT_DIR / "audiosep_chunk_validation.json").write_text(
    json.dumps(AUDIOSEP_CHUNK_VALIDATION, indent=2), encoding="utf-8"
)

# Structured human listening scorecard template
scorecard_rows = []
for rec in ALL_REAL_RESULTS:
    scorecard_rows.append({
        "track": "REAL_RECORDING",
        "interference_source": "real_recording_unknown",
        "scenario_id": "real_ukulele_upload",
        "engine": rec["engine"],
        "query_id": rec["query_id"],
        "ukulele_recognizable_1to5": "",
        "strum_attacks_decay_1to5": "",
        "background_suppression_1to5": "",
        "voices_or_traffic_still_audible": "",
        "artifacts_metallic_watery_pumping": "",
        "listener_notes": "",
    })
for sid in PREVIEW_SCENARIOS:
    sc = next((s for s in SYNTHETIC_SCENARIOS if s["scenario_id"] == sid), None)
    if sc is None:
        continue
    for rec in [r for r in ALL_SYNTH_RESULTS if r["scenario_id"] == sid and r["query_id"] in PREVIEW_QUERIES]:
        scorecard_rows.append({
            "track": "CALIBRATED_0DB",
            "interference_source": sc["interference_source"],
            "scenario_id": sid,
            "engine": rec["engine"],
            "query_id": rec["query_id"],
            "ukulele_recognizable_1to5": "",
            "strum_attacks_decay_1to5": "",
            "background_suppression_1to5": "",
            "voices_or_traffic_still_audible": "",
            "artifacts_metallic_watery_pumping": "",
            "listener_notes": "",
        })
pd.DataFrame(scorecard_rows).to_csv(REPORT_DIR / "listening_scorecard_template.csv", index=False)

# Evidence-gated Markdown summary report
md_lines = [
    "# SONORA Stage 5 — Cloud Feasibility Benchmark Report (CLAPSep vs. AudioSep)",
    "",
    "## 1. Execution Hardware & Checkpoint Load Security Audit",
    "```json",
    json.dumps(HARDWARE_INFO, indent=2),
    "```",
    "",
    "### Model Load & Peak Memory Telemetry",
    _safe_md_table(df_load),
    "",
    "### Checkpoint Deserialization Audit Log",
    "```json",
    json.dumps(CHECKPOINT_LOAD_AUDIT_LOG, indent=2),
    "```",
    "",
    "### AudioSep Short/Long Chunk Validation",
    "```json",
    json.dumps(AUDIOSEP_CHUNK_VALIDATION, indent=2),
    "```",
    "",
    "## 2. Clean-Ukulele Regression Guard (No Background Noise)",
    _safe_md_table(df_clean),
    "",
    "## 3. Calibrated Noisy Mixtures Summary (Distinguishing Real Speech vs. Synthetic Proxies)",
    _safe_md_table(summary_by_query),
    "",
    "## 4. Best Query Per Engine by Noise Type & SNR",
    _safe_md_table(best_by_noise),
    "",
    "## 5. Real Ukulele Recording Track",
    (
        _safe_md_table(df_real[real_cols])
        if not df_real.empty
        else "_No real ukulele recording was uploaded in Section 4. Missing Stage 4 output was NOT fabricated._"
    ),
    "",
    "## 6. Evidence-Gated Decision Questions (Complete After Listening Evaluation)",
    "Complete using BOTH objective tables above AND `listening_scorecard_template.csv`. Do not claim perfect separation.",
    "1. **Is CLAPSep genuinely effective for ukulele extraction?**",
    "2. **Is AudioSep better or worse than CLAPSep (and do CLAPSep's negative queries measurably help)?**",
    "3. **Which model preserves ukulele sound more naturally (timbre, strum attacks, decay, level retention)?**",
    "4. **Which unwanted sounds remain (real speech vs. synthetic speech, traffic, fan, wind, pink noise, acoustic guitar)?**",
    "5. **Is either model suitable for eventual SONORA integration (given 32 kHz mono cap and local 8 GB CPU-only limits)?**",
    "6. **What are the verified limitations and residual security/licensing considerations?**",
]
report_md_path = REPORT_DIR / "stage5_cloud_benchmark_report.md"
report_md_path.write_text("\n".join(md_lines), encoding="utf-8")

# Package downloadable ZIP bundle (all manifests, reports, real audio, and 0 dB calibrated comparisons)
zip_path = WORK_ROOT / "sonora_stage5_cloud_benchmark_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [
        WORK_ROOT / "environment_manifest.json",
        WORK_ROOT / "checkpoint_manifest.json",
        SYNTH_DIR / "synthetic_manifest.json",
        *REPORT_DIR.glob("*"),
    ]:
        if p.exists() and p.is_file():
            zf.write(p, arcname=str(p.relative_to(WORK_ROOT)))

    if REAL_RECORDING_INFO is not None:
        for p in REAL_INPUT_DIR.glob("*"):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(WORK_ROOT)))
        for rec in ALL_REAL_RESULTS:
            for k in ("audition_wav", "residual_diff_wav"):
                fp = Path(rec[k])
                if fp.exists():
                    zf.write(fp, arcname=str(fp.relative_to(WORK_ROOT)))

    if REAL_SPEECH_INFO is not None:
        sp_manifest = REAL_SPEECH_DIR / "real_speech_manifest.json"
        if sp_manifest.exists():
            zf.write(sp_manifest, arcname=str(sp_manifest.relative_to(WORK_ROOT)))

    for sid in PREVIEW_SCENARIOS:
        sc = next((s for s in SYNTHETIC_SCENARIOS if s["scenario_id"] == sid), None)
        if sc:
            for k in ("clean_path", "noise_path", "mixture_path"):
                if sc.get(k):
                    fp = Path(sc[k])
                    if fp.exists():
                        zf.write(fp, arcname=str(fp.relative_to(WORK_ROOT)))
        for rec in [r for r in ALL_SYNTH_RESULTS if r["scenario_id"] == sid and r["query_id"] in PREVIEW_QUERIES]:
            for k in ("audition_wav", "residual_diff_wav"):
                fp = Path(rec[k])
                if fp.exists():
                    zf.write(fp, arcname=str(fp.relative_to(WORK_ROOT)))

print(f"\n[OK] Saved Markdown report: {report_md_path}")
print(f"[OK] Saved downloadable ZIP bundle ({zip_path.stat().st_size / (1024 ** 2):.2f} MB): {zip_path}")

try:
    from google.colab import files
    print("Triggering browser download of benchmark results ZIP...")
    files.download(str(zip_path))
except ImportError:
    print("Download the ZIP manually from:", zip_path)